In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

# -----------------------------
# 1. Basic settings
# -----------------------------
n = 1500

departments = [
    "IT", "Sales", "HR", "Finance",
    "Marketing", "Operations", "Customer Support"
]

cities = [
    "Islamabad", "Lahore", "Karachi",
    "Rawalpindi", "Peshawar", "Multan"
]

genders = ["Male", "Female"]

job_titles = {
    "IT": [
        "Software Engineer",
        "Data Analyst",
        "IT Support Specialist",
        "IT Manager"
    ],
    "Sales": [
        "Sales Executive",
        "Sales Manager",
        "Account Executive"
    ],
    "HR": [
        "HR Specialist",
        "HR Manager",
        "Recruiter"
    ],
    "Finance": [
        "Financial Analyst",
        "Accountant",
        "Finance Manager"
    ],
    "Marketing": [
        "Marketing Specialist",
        "Marketing Manager",
        "Content Strategist"
    ],
    "Operations": [
        "Operations Analyst",
        "Operations Manager",
        "Coordinator"
    ],
    "Customer Support": [
        "Support Representative",
        "Support Specialist",
        "Support Manager"
    ]
}

# -----------------------------
# 2. Generate employee data
# -----------------------------

employee_id = [
    f"E{10001 + i}"
    for i in range(n)
]

name = [
    f"Employee_{i+1}"
    for i in range(n)
]

gender = np.random.choice(
    genders,
    n,
    p=[0.55, 0.45]
)

department = np.random.choice(
    departments,
    n,
    p=[0.20, 0.18, 0.08, 0.12, 0.10, 0.18, 0.14]
)

# Age
age = np.random.normal(34, 8, n)
age = np.round(age)
age = np.clip(age, 21, 60)

# Experience
years_experience = (
    (age - 22) *
    np.random.uniform(0.45, 0.75, n)
)

years_experience = np.maximum(
    0,
    np.round(years_experience, 1)
)

# Job title based on department
job_title = [
    np.random.choice(job_titles[d])
    for d in department
]

# Salary
salary = (
    45000
    + years_experience * 8500
    + np.random.normal(0, 18000, n)
)

salary = np.clip(
    salary,
    35000,
    450000
)

salary = np.round(salary, -2)

# Join date
start_date = pd.Timestamp("2017-01-01")
end_date = pd.Timestamp("2025-12-31")

days_range = (
    end_date - start_date
).days

random_days = np.random.randint(
    0,
    days_range,
    n
)

join_date = (
    start_date +
    pd.to_timedelta(random_days, unit="D")
)

# Other columns
city = np.random.choice(
    cities,
    n
)

performance_score = np.random.choice(
    [1, 2, 3, 4, 5],
    n,
    p=[0.05, 0.15, 0.35, 0.32, 0.13]
)

satisfaction_score = np.random.choice(
    [1, 2, 3, 4, 5],
    n,
    p=[0.06, 0.12, 0.30, 0.34, 0.18]
)

overtime = np.random.choice(
    ["Yes", "No"],
    n,
    p=[0.30, 0.70]
)

remote_work = np.random.choice(
    ["Yes", "No"],
    n,
    p=[0.38, 0.62]
)

projects_completed = np.maximum(
    0,
    np.round(
        np.random.normal(5, 2.5, n)
    )
).astype(int)

training_hours = np.maximum(
    0,
    np.round(
        np.random.normal(24, 12, n)
    )
).astype(int)

leave_days = np.maximum(
    0,
    np.round(
        np.random.normal(12, 5, n)
    )
).astype(int)

# -----------------------------
# 3. Attrition
# -----------------------------

attrition_probability = (
    0.08
    + np.where(overtime == "Yes", 0.07, 0)
    + np.where(satisfaction_score <= 2, 0.08, 0)
    - np.where(satisfaction_score >= 4, 0.03, 0)
)

attrition_probability = np.clip(
    attrition_probability,
    0.02,
    0.35
)

attrition = np.where(
    np.random.random(n) < attrition_probability,
    "Yes",
    "No"
)

# -----------------------------
# 4. Promotion
# -----------------------------

promotion = np.where(
    (
        (years_experience >= 4)
        & (performance_score >= 4)
        & (np.random.random(n) < 0.55)
    ),
    "Yes",
    "No"
)

# -----------------------------
# 5. Create DataFrame
# -----------------------------

df = pd.DataFrame({
    "Employee_ID": employee_id,
    "Name": name,
    "Age": age,
    "Gender": gender,
    "Department": department,
    "Job_Title": job_title,
    "Salary": salary,
    "Years_Experience": years_experience,
    "Join_Date": join_date,
    "City": city,
    "Performance_Score": performance_score,
    "Satisfaction_Score": satisfaction_score,
    "Overtime": overtime,
    "Remote_Work": remote_work,
    "Projects_Completed": projects_completed,
    "Training_Hours": training_hours,
    "Leave_Days": leave_days,
    "Promotion": promotion,
    "Attrition": attrition
})

# -----------------------------
# 6. Add missing values
# -----------------------------

missing_values = {
    "Age": 35,
    "Salary": 25,
    "Performance_Score": 20,
    "Satisfaction_Score": 18,
    "Join_Date": 22,
    "City": 15
}

for column, count in missing_values.items():

    indexes = np.random.choice(
        df.index,
        count,
        replace=False
    )

    df.loc[indexes, column] = np.nan

# -----------------------------
# 7. Create text inconsistencies
# -----------------------------

indexes = np.random.choice(
    df.index,
    30,
    replace=False
)

# Lowercase departments
df.loc[
    indexes[:10],
    "Department"
] = (
    df.loc[indexes[:10], "Department"]
    .str.lower()
)

# Extra spaces
df.loc[
    indexes[10:20],
    "Department"
] = (
    " "
    + df.loc[indexes[10:20], "Department"]
    + " "
)

# Lowercase gender
df.loc[
    indexes[20:],
    "Gender"
] = (
    df.loc[indexes[20:], "Gender"]
    .str.lower()
)

# -----------------------------
# 8. Make Salary messy
# -----------------------------

salary_indexes = np.random.choice(
    df.index[df["Salary"].notna()],
    12,
    replace=False
)

# Convert some values to strings
df["Salary"] = df["Salary"].astype("object")

df.loc[
    salary_indexes[:5],
    "Salary"
] = (
    df.loc[salary_indexes[:5], "Salary"]
    .astype(str)
)

# Add invalid values
df.loc[
    salary_indexes[5:8],
    "Salary"
] = "N/A"

df.loc[
    salary_indexes[8:],
    "Salary"
] = "unknown"

# -----------------------------
# 9. Add salary outliers
# -----------------------------

outlier_indexes = np.random.choice(
    df.index[df["Salary"].notna()],
    5,
    replace=False
)

df.loc[
    outlier_indexes,
    "Salary"
] = [
    750000,
    680000,
    920000,
    1200000,
    850000
]

# -----------------------------
# 10. Add duplicate rows
# -----------------------------

duplicates = df.sample(
    15,
    random_state=7
)

df = pd.concat(
    [df, duplicates],
    ignore_index=True
)

# -----------------------------
# 11. Shuffle dataset
# -----------------------------

df = df.sample(
    frac=1,
    random_state=99
).reset_index(drop=True)

# -----------------------------
# 12. Save CSV
# -----------------------------

file_path = Path(
    "employee_hr_analytics.csv"
)

df.to_csv(
    file_path,
    index=False
)

print("Dataset created successfully!")
print("File:", file_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset created successfully!
File: employee_hr_analytics.csv
Rows: 1515
Columns: 19


In [1]:
import pandas as pd

df = pd.read_csv("data/employee_hr_analytics.csv")

In [2]:
df.columns

Index(['Employee_ID', 'Name', 'Age', 'Gender', 'Department', 'Job_Title',
       'Salary', 'Years_Experience', 'Join_Date', 'City', 'Performance_Score',
       'Satisfaction_Score', 'Overtime', 'Remote_Work', 'Projects_Completed',
       'Training_Hours', 'Leave_Days', 'Promotion', 'Attrition'],
      dtype='str')

In [3]:
df.shape

(1515, 19)

In [5]:
df.dtypes

Employee_ID               str
Name                      str
Age                   float64
Gender                    str
Department                str
Job_Title                 str
Salary                    str
Years_Experience      float64
Join_Date                 str
City                      str
Performance_Score     float64
Satisfaction_Score    float64
Overtime                  str
Remote_Work               str
Projects_Completed      int64
Training_Hours          int64
Leave_Days              int64
Promotion                 str
Attrition                 str
dtype: object

In [6]:
df.isna().sum()

Employee_ID            0
Name                   0
Age                   35
Gender                 0
Department             0
Job_Title              0
Salary                28
Years_Experience       0
Join_Date             22
City                  15
Performance_Score     20
Satisfaction_Score    18
Overtime               0
Remote_Work            0
Projects_Completed     0
Training_Hours         0
Leave_Days             0
Promotion              0
Attrition              0
dtype: int64

In [7]:
df.head(5)

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,Performance_Score,Satisfaction_Score,Overtime,Remote_Work,Projects_Completed,Training_Hours,Leave_Days,Promotion,Attrition
0,E10674,Employee_674,43.0,Male,Operations,Coordinator,150100.0,13.9,2024-04-21,Peshawar,4.0,3.0,No,Yes,4,33,8,Yes,No
1,E10204,Employee_204,NaN,Female,Finance,Finance Manager,142800.0,9.0,2021-07-31,Rawalpindi,3.0,3.0,No,No,7,12,9,No,Yes
2,E10308,Employee_308,41.0,Male,IT,Software Engineer,NaN,10.1,2017-06-09,Peshawar,5.0,2.0,No,No,6,31,12,Yes,No
3,E11333,Employee_1333,35.0,Male,IT,IT Support Specialist,130000.0,8.9,2019-11-14,Islamabad,3.0,4.0,No,No,9,20,16,No,No
4,E10492,Employee_492,22.0,Female,IT,Data Analyst,35000.0,0.0,2023-01-16,Islamabad,2.0,3.0,No,Yes,5,39,12,No,No


In [8]:
# Find the duplicate employees

In [9]:
df.duplicated().sum()

np.int64(15)

In [10]:
df[df.duplicated()]

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,Performance_Score,Satisfaction_Score,Overtime,Remote_Work,Projects_Completed,Training_Hours,Leave_Days,Promotion,Attrition
251,E10548,Employee_548,48.0,Female,Operations,Operations Manager,174800.0,17.3,2019-06-25,Lahore,3.0,4.0,No,Yes,0,27,11,No,No
395,E10639,Employee_639,25.0,Male,IT,Data Analyst,43700.0,2.1,2022-10-25,Lahore,3.0,3.0,No,No,3,10,17,No,No
473,E10836,Employee_836,45.0,Male,HR,HR Specialist,142300.0,12.4,2024-05-07,Lahore,3.0,4.0,Yes,No,2,31,6,No,Yes
548,E10718,Employee_718,34.0,Female,Sales,Sales Executive,104300.0,8.0,2024-12-01,Lahore,5.0,4.0,Yes,No,0,22,9,No,No
869,E10417,Employee_417,34.0,Male,Customer Support,Support Specialist,115800.0,6.9,2020-08-20,Multan,5.0,4.0,No,No,4,44,25,Yes,No
918,E11159,Employee_1159,38.0,Female,Sales,Sales Executive,97300.0,10.0,2017-06-11,Islamabad,2.0,2.0,Yes,Yes,2,45,21,No,No
990,E10979,Employee_979,25.0,Male,IT,IT Support Specialist,72900.0,1.8,2021-08-05,Multan,1.0,3.0,No,No,0,25,15,No,No
1006,E11022,Employee_1022,24.0,Male,IT,IT Manager,66700.0,1.1,2024-10-04,Lahore,4.0,5.0,Yes,Yes,2,21,14,No,Yes
1081,E11150,Employee_1150,31.0,Female,hr,HR Specialist,103100.0,5.0,2022-10-24,Lahore,4.0,4.0,No,Yes,2,16,18,Yes,No
1087,E10472,Employee_472,41.0,Female,Operations,Operations Analyst,173000.0,11.6,2019-12-07,Multan,3.0,4.0,No,No,6,32,13,No,No


In [11]:
# Clean the duplicates

In [12]:
df=df.drop_duplicates(keep="first")

In [13]:
df.shape

(1500, 19)

In [14]:
# Investigate missing values
df.isna().sum()

Employee_ID            0
Name                   0
Age                   35
Gender                 0
Department             0
Job_Title              0
Salary                28
Years_Experience       0
Join_Date             22
City                  15
Performance_Score     20
Satisfaction_Score    18
Overtime               0
Remote_Work            0
Projects_Completed     0
Training_Hours         0
Leave_Days             0
Promotion              0
Attrition              0
dtype: int64

In [15]:
# "What should we do with the 35 missing ages?"

# Age missing values:

In [16]:
df["Age"].describe()

count    1465.000000
mean       34.153584
std         7.729213
min        21.000000
25%        28.000000
50%        34.000000
75%        39.000000
max        60.000000
Name: Age, dtype: float64

In [17]:
df.groupby("Gender")["Age"].mean()

Gender
Female    34.063158
Male      34.236709
female    33.800000
male      33.400000
Name: Age, dtype: float64

In [18]:
df["Gender"]=df["Gender"].replace({
    "female":"Female",
    "male":"Male"}
)

In [19]:
df.groupby("Gender")["Age"].median()

Gender
Female    34.0
Male      34.0
Name: Age, dtype: float64

In [20]:
df["Age"]=df["Age"].fillna(df.groupby("Gender")["Age"].transform("median"))

In [21]:
df["Age"].isna().sum()

np.int64(0)

# Salary missing values:

In [22]:
df["Salary"].isna().sum()

np.int64(28)

In [23]:
df["Salary"].describe()

count        1472
unique        942
top       35000.0
freq           57
Name: Salary, dtype: object

In [24]:
df["Salary"].unique()

<StringArray>
['150100.0', '142800.0',        nan, '130000.0',  '35000.0',  '96800.0',
 '111100.0',  '53500.0', '110300.0',  '67000.0',
 ...
  '62900.0',  '77000.0', '147400.0', '170500.0',  '40600.0',  '92100.0',
  '83300.0', '193900.0', '166200.0',  '51600.0']
Length: 943, dtype: str

In [25]:
pd.to_numeric(df["Salary"],errors="coerce").isna().sum()

np.int64(32)

In [26]:
df.loc[pd.to_numeric(df["Salary"],errors="coerce").isna(),"Salary"].value_counts()

Salary
unknown    4
Name: count, dtype: int64

In [27]:
df["Salary"]=pd.to_numeric(df["Salary"],errors="coerce")

In [28]:
df["Salary"].dtype

dtype('float64')

In [29]:
df["Salary"].describe()

count    1.468000e+03
mean     1.106646e+05
std      6.390653e+04
min      3.500000e+04
25%      7.370000e+04
50%      1.062500e+05
75%      1.362500e+05
max      1.200000e+06
Name: Salary, dtype: float64

In [30]:
df["Salary"].mean()

np.float64(110664.64577656676)

In [31]:
df["Salary"].nlargest(10)

1067    1200000.0
1144     920000.0
490      850000.0
927      750000.0
1284     680000.0
420      275600.0
818      265200.0
752      256600.0
1078     239400.0
1069     239000.0
Name: Salary, dtype: float64

In [32]:
df.groupby("Job_Title")["Salary"].median().sort_values()

Job_Title
Support Representative     85200.0
HR Manager                 95200.0
Recruiter                  98800.0
IT Manager                101250.0
Software Engineer         101800.0
Financial Analyst         102300.0
Coordinator               104000.0
IT Support Specialist     104800.0
Accountant                104900.0
Account Executive         104950.0
Sales Executive           106100.0
Support Manager           106150.0
Operations Manager        106450.0
Data Analyst              106850.0
Operations Analyst        107300.0
Support Specialist        107400.0
Marketing Manager         108650.0
Marketing Specialist      110850.0
Sales Manager             111050.0
Finance Manager           113500.0
Content Strategist        115600.0
HR Specialist             116300.0
Name: Salary, dtype: float64

In [33]:
df["Salary"]=df["Salary"].fillna(df.groupby("Job_Title")["Salary"].transform("median"))

In [34]:
df["Salary"].isna().sum()

np.int64(0)

# Join_Date missing values

In [35]:
df["Join_Date"].isna().sum()

np.int64(22)

In [36]:
df["Join_Date"].describe()

count           1478
unique          1193
top       2019-12-31
freq               4
Name: Join_Date, dtype: object

In [37]:
df["Join_Date"]=pd.to_datetime(df["Join_Date"],errors="coerce")

In [38]:
df["Join_Date"].describe()

count                          1478
mean     2021-06-10 10:38:09.580514
min             2017-01-01 00:00:00
25%             2019-04-29 00:00:00
50%             2021-05-06 12:00:00
75%             2023-08-19 06:00:00
max             2025-12-30 00:00:00
Name: Join_Date, dtype: object

In [39]:
df[df["Join_Date"].isna()]["Department"].value_counts()

Department
Sales               6
Operations          4
Customer Support    3
HR                  3
Finance             2
Marketing           2
IT                  2
Name: count, dtype: int64

In [40]:
df["Join_Date"].isna().sum()

np.int64(22)

# City missing values:

In [41]:
df["City"].isna().sum()

np.int64(15)

In [42]:
df["City"].describe()

count       1485
unique         6
top       Lahore
freq         260
Name: City, dtype: object

In [43]:
df["City"].value_counts(dropna=False)

City
Lahore        260
Multan        258
Rawalpindi    257
Peshawar      238
Islamabad     236
Karachi       236
NaN            15
Name: count, dtype: int64

In [44]:
df[df["City"].isna()]["Department"].value_counts()

Department
Operations          3
HR                  3
Sales               2
Marketing           2
Customer Support    2
Finance             2
IT                  1
Name: count, dtype: int64

In [45]:
df["City"].isna().sum()

np.int64(15)

# Performance_Score missing values:

In [46]:
df["Performance_Score"].isna().sum()

np.int64(20)

In [47]:
df["Performance_Score"].describe()

count    1480.000000
mean        3.340541
std         1.029112
min         1.000000
25%         3.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: Performance_Score, dtype: float64

In [48]:
df.groupby("Department")["Performance_Score"].median()

Department
 HR                 4.0
 IT                 3.5
 Marketing          4.0
 Operations         4.0
 Sales              4.0
Customer Support    3.0
Finance             3.0
HR                  3.0
IT                  3.0
Marketing           3.0
Operations          4.0
Sales               3.0
customer support    2.0
finance             3.0
hr                  4.0
it                  4.0
marketing           3.0
operations          NaN
sales               3.0
Name: Performance_Score, dtype: float64

In [49]:
df["Department"]=df["Department"].str.strip().str.title()

In [50]:
df.groupby("Department")["Performance_Score"].median()

Department
Customer Support    3.0
Finance             3.0
Hr                  3.0
It                  3.0
Marketing           3.0
Operations          4.0
Sales               3.0
Name: Performance_Score, dtype: float64

In [51]:
df["Performance_Score"]=df["Performance_Score"].fillna(df.groupby("Department")["Performance_Score"].transform("median"))

In [52]:
df["Performance_Score"].isna().sum()

np.int64(0)

# Satisfaction_Score missing values:

In [53]:
df["Satisfaction_Score"].isna().sum()

np.int64(18)

In [54]:
df["Satisfaction_Score"].describe()

count    1482.000000
mean        3.460189
std         1.085834
min         1.000000
25%         3.000000
50%         4.000000
75%         4.000000
max         5.000000
Name: Satisfaction_Score, dtype: float64

In [55]:
df.groupby("Department")["Satisfaction_Score"].median()

Department
Customer Support    3.0
Finance             3.0
Hr                  4.0
It                  3.0
Marketing           4.0
Operations          4.0
Sales               3.0
Name: Satisfaction_Score, dtype: float64

In [56]:
df["Satisfaction_Score"] = df["Satisfaction_Score"].fillna(
    df.groupby("Department")["Satisfaction_Score"].transform("median")
)

In [57]:
df["Satisfaction_Score"].isna().sum()

np.int64(0)

In [58]:
df.isna().sum()

Employee_ID            0
Name                   0
Age                    0
Gender                 0
Department             0
Job_Title              0
Salary                 0
Years_Experience       0
Join_Date             22
City                  15
Performance_Score      0
Satisfaction_Score     0
Overtime               0
Remote_Work            0
Projects_Completed     0
Training_Hours         0
Leave_Days             0
Promotion              0
Attrition              0
dtype: int64

In [59]:
df["Department"] = df["Department"].replace({
    "Hr": "HR",
    "It": "IT"
})

In [60]:
df["Department"].unique()

<StringArray>
['Operations', 'Finance', 'IT', 'Sales', 'Marketing', 'Customer Support',
 'HR']
Length: 7, dtype: str

In [61]:
df["Gender"].value_counts(dropna=False)

Gender
Male      815
Female    685
Name: count, dtype: int64

In [62]:
df["Job_Title"].value_counts()

Job_Title
Operations Analyst        102
Sales Manager              96
Data Analyst               91
Sales Executive            91
Operations Manager         90
Coordinator                88
Account Executive          76
IT Support Specialist      75
Support Specialist         72
Software Engineer          71
Support Representative     69
Financial Analyst          66
Accountant                 66
IT Manager                 61
Support Manager            61
Content Strategist         58
Marketing Specialist       55
Recruiter                  50
Finance Manager            46
Marketing Manager          41
HR Manager                 38
HR Specialist              37
Name: count, dtype: int64

In [63]:
df["City"].value_counts(dropna=False)

City
Lahore        260
Multan        258
Rawalpindi    257
Peshawar      238
Islamabad     236
Karachi       236
NaN            15
Name: count, dtype: int64

In [64]:
df["Overtime"].value_counts(dropna=False)

Overtime
No     1071
Yes     429
Name: count, dtype: int64

In [65]:
df["Remote_Work"].value_counts(dropna=False)

Remote_Work
No     914
Yes    586
Name: count, dtype: int64

In [66]:
df["Promotion"].value_counts(dropna=False)

Promotion
No     1215
Yes     285
Name: count, dtype: int64

In [67]:
df["Attrition"].value_counts(dropna=False)

Attrition
No     1347
Yes     153
Name: count, dtype: int64

In [68]:
df.duplicated().sum()

np.int64(0)

# EDA

In [69]:
# "How many employees are in each department?"

In [70]:
df.groupby("Department")["Name"].count()

Department
Customer Support    202
Finance             178
HR                  125
IT                  298
Marketing           154
Operations          280
Sales               263
Name: Name, dtype: int64

In [71]:
# What is the average salary in each department?

In [72]:
df.groupby("Department")["Salary"].mean()

Department
Customer Support    103844.801980
Finance             115277.528090
HR                  104615.200000
IT                  107133.221477
Marketing           117392.857143
Operations          113538.214286
Sales               112100.760456
Name: Salary, dtype: float64

In [73]:
# What is the median salary in each department?

In [74]:
df.groupby("Department")["Salary"].median()

Department
Customer Support    101350.0
Finance             109850.0
HR                  100400.0
IT                  103400.0
Marketing           112500.0
Operations          106450.0
Sales               106900.0
Name: Salary, dtype: float64

In [75]:
# What is the average Years_Experience for each department?

In [76]:
df.groupby("Department")["Years_Experience"].mean()

Department
Customer Support    6.843069
Finance             7.964607
HR                  7.054400
IT                  7.288255
Marketing           7.737013
Operations          7.128929
Sales               7.297338
Name: Years_Experience, dtype: float64

In [77]:
# What is the median Years_Experience for each department?
df.groupby("Department")["Years_Experience"].median()

Department
Customer Support    6.65
Finance             7.85
HR                  7.00
IT                  6.80
Marketing           7.45
Operations          6.65
Sales               6.90
Name: Years_Experience, dtype: float64

In [78]:
# Which job title has the highest average salary?
df.groupby("Job_Title")["Salary"].mean().nlargest()

Job_Title
Marketing Manager    129525.609756
Coordinator          124098.863636
Finance Manager      123723.913043
Account Executive    120481.578947
Accountant           115556.060606
Name: Salary, dtype: float64

In [79]:
# What is the average Performance_Score for each department?
df.groupby("Department")["Performance_Score"].mean()

Department
Customer Support    3.420792
Finance             3.264045
HR                  3.144000
IT                  3.385906
Marketing           3.227273
Operations          3.453571
Sales               3.311787
Name: Performance_Score, dtype: float64

In [80]:
# What is the average Satisfaction_Score for each department?
df.groupby("Department")["Satisfaction_Score"].mean()

Department
Customer Support    3.415842
Finance             3.398876
HR                  3.600000
IT                  3.442953
Marketing           3.493506
Operations          3.485714
Sales               3.433460
Name: Satisfaction_Score, dtype: float64

In [81]:
# What is the average salary for each Job_Title, and show the results from highest salary to lowest?
df.groupby("Job_Title")["Salary"].mean().sort_values(ascending=False)

Job_Title
Marketing Manager         129525.609756
Coordinator               124098.863636
Finance Manager           123723.913043
Account Executive         120481.578947
Accountant                115556.060606
Content Strategist        114182.758621
HR Specialist             112978.378378
Support Manager           112336.885246
Sales Manager             111975.000000
Marketing Specialist      111733.636364
Operations Manager        111222.222222
Data Analyst              110902.747253
Financial Analyst         109112.121212
Operations Analyst        106470.588235
Support Specialist        105840.277778
IT Support Specialist     105828.000000
Software Engineer         105522.535211
Sales Executive           105234.065934
IT Manager                104989.344262
Recruiter                 103636.000000
HR Manager                 97760.526316
Support Representative     94255.072464
Name: Salary, dtype: float64

In [82]:
# Which department has the highest average Performance_Score?
df.groupby("Department")["Performance_Score"].mean().nlargest(1)

Department
Operations    3.453571
Name: Performance_Score, dtype: float64

In [83]:
df.groupby("Department")["Salary"].mean().nlargest(1)

Department
Marketing    117392.857143
Name: Salary, dtype: float64

In [84]:
df.groupby("Department")["Employee_ID"].count()

Department
Customer Support    202
Finance             178
HR                  125
IT                  298
Marketing           154
Operations          280
Sales               263
Name: Employee_ID, dtype: int64

In [85]:
# What is the number of employees who left (Attrition = "Yes") in each department?
df[df["Attrition"]=="Yes"]["Department"].value_counts()

Department
IT                  35
Sales               32
Operations          30
Finance             21
Customer Support    21
Marketing            9
HR                   5
Name: count, dtype: int64

In [86]:
# What percentage of employees in each department have left the company?
Attrition_avg=df[df["Attrition"]=="Yes"]["Department"].value_counts()

In [87]:
Total_employee=df.groupby("Department")["Name"].count()
Total_employee

Department
Customer Support    202
Finance             178
HR                  125
IT                  298
Marketing           154
Operations          280
Sales               263
Name: Name, dtype: int64

In [88]:
Percentage_Attrition=(Attrition_avg/Total_employee)*100

In [89]:
Percentage_Attrition

Department
Customer Support    10.396040
Finance             11.797753
HR                   4.000000
IT                  11.744966
Marketing            5.844156
Operations          10.714286
Sales               12.167300
dtype: float64

In [90]:
# What is the attrition rate for employees who work overtime vs employees who don't?
# Does overtime relate to employee attrition?

In [91]:
Attrition_overtime=df[df["Attrition"]=="Yes"]["Overtime"].value_counts()

In [92]:
Total_overtime=df["Overtime"].value_counts()

In [93]:
Attrition_rate=(Attrition_overtime/Total_overtime)*100

In [94]:
Attrition_rate

Overtime
No      8.870215
Yes    13.519814
Name: count, dtype: float64

In [95]:
# Is overtime associated with lower employee satisfaction?

In [96]:
df.groupby("Overtime")["Satisfaction_Score"].mean()

Overtime
No     3.453782
Yes    3.470862
Name: Satisfaction_Score, dtype: float64

In [97]:
# Is performance associated with promotion?
df.groupby("Promotion")["Performance_Score"].mean()

Promotion
No     3.125926
Yes    4.249123
Name: Performance_Score, dtype: float64

In [98]:
df.groupby("Department")["Promotion"].value_counts()

Department        Promotion
Customer Support  No           168
                  Yes           34
Finance           No           146
                  Yes           32
HR                No           107
                  Yes           18
IT                No           239
                  Yes           59
Marketing         No           123
                  Yes           31
Operations        No           216
                  Yes           64
Sales             No           216
                  Yes           47
Name: count, dtype: int64

In [99]:
# What percentage of employees in each department received a promotion?
Promotion=df[df["Promotion"]=="Yes"]["Department"].value_counts()

In [100]:
total=df["Department"].value_counts()

In [101]:
Promotion_perc=(Promotion/total)*100
Promotion_perc

Department
Customer Support    16.831683
Finance             17.977528
HR                  14.400000
IT                  19.798658
Marketing           20.129870
Operations          22.857143
Sales               17.870722
Name: count, dtype: float64

In [102]:
# Among employees who received a promotion, what is their average salary in each department?
df.groupby(["Department","Promotion"])["Salary"].mean()


Department        Promotion
Customer Support  No            99290.178571
                  Yes          126350.000000
Finance           No           112308.219178
                  Yes          128825.000000
HR                No           102196.261682
                  Yes          118994.444444
IT                No           103555.230126
                  Yes          121627.118644
Marketing         No           114852.032520
                  Yes          127474.193548
Operations        No           107325.462963
                  Yes          134506.250000
Sales             No           110422.685185
                  Yes          119812.765957
Name: Salary, dtype: float64

In [103]:
promoted_employees=df[df["Promotion"]=="Yes"]

In [104]:
promoted_employees.groupby("Department")["Salary"].mean()

Department
Customer Support    126350.000000
Finance             128825.000000
HR                  118994.444444
IT                  121627.118644
Marketing           127474.193548
Operations          134506.250000
Sales               119812.765957
Name: Salary, dtype: float64

In [105]:
# For each department, compare the average salary of promoted employees vs non-promoted employees.


In [106]:
avg_salary=df.groupby(["Promotion","Department"])["Salary"].mean()

In [107]:
avg_salary

Promotion  Department      
No         Customer Support     99290.178571
           Finance             112308.219178
           HR                  102196.261682
           IT                  103555.230126
           Marketing           114852.032520
           Operations          107325.462963
           Sales               110422.685185
Yes        Customer Support    126350.000000
           Finance             128825.000000
           HR                  118994.444444
           IT                  121627.118644
           Marketing           127474.193548
           Operations          134506.250000
           Sales               119812.765957
Name: Salary, dtype: float64

In [108]:
df.groupby("Department").agg(
    avg_salary=("Salary", "mean"),
    highest_salary=("Salary", "max"),
    avg_performance=("Performance_Score", "mean"),
    employee_count=("Name", "count")
)

,avg_salary,highest_salary,avg_performance,employee_count
Department,,,,
Customer Support,103844.801980,236400.0,3.420792,202
Finance,115277.528090,680000.0,3.264045,178
HR,104615.200000,200700.0,3.144000,125
IT,107133.221477,239400.0,3.385906,298
Marketing,117392.857143,850000.0,3.227273,154
Operations,113538.214286,920000.0,3.453571,280
Sales,112100.760456,1200000.0,3.311787,263


In [109]:
df.groupby("Department").agg({
    "Salary":["mean","max"],
    "Performance_Score":"mean",
    "Name":"count"
})

Salary            Performance_Score  Name
                           mean        max              mean count
Department                                                        
Customer Support  103844.801980   236400.0          3.420792   202
Finance           115277.528090   680000.0          3.264045   178
HR                104615.200000   200700.0          3.144000   125
IT                107133.221477   239400.0          3.385906   298
Marketing         117392.857143   850000.0          3.227273   154
Operations        113538.214286   920000.0          3.453571   280
Sales             112100.760456  1200000.0          3.311787   263

In [110]:
# Which department has the highest average salary among overtime employees?

In [111]:
work_overtime=df[df["Overtime"]=="Yes"].groupby("Department")["Salary"].mean().nlargest(1)

In [112]:
work_overtime

Department
Marketing    135041.489362
Name: Salary, dtype: float64

In [113]:
# Which department has the highest average satisfaction score among employees who have attrition = Yes?

In [114]:
df[df["Attrition"]=="Yes"].groupby("Department")["Satisfaction_Score"].mean().nlargest(1)

Department
HR    3.4
Name: Satisfaction_Score, dtype: float64

In [115]:
# average Satisfaction_Score for Attrition = Yes by Department, and then sort all departments from highest to lowest.

In [116]:
df[df["Attrition"]=="Yes"].groupby("Department")["Satisfaction_Score"].mean().sort_values(ascending=False)

Department
HR                  3.400000
Operations          3.366667
Finance             3.285714
Sales               3.031250
IT                  2.885714
Customer Support    2.857143
Marketing           2.666667
Name: Satisfaction_Score, dtype: float64

In [117]:
# For each department, calculate the attrition rate separately for employees who work overtime and employees who don't.

In [118]:
df[df["Attrition"] == "Yes"].groupby(["Department", "Overtime"]).size()

Department        Overtime
Customer Support  No          13
                  Yes          8
Finance           No          16
                  Yes          5
HR                No           2
                  Yes          3
IT                No          19
                  Yes         16
Marketing         No           7
                  Yes          2
Operations        No          18
                  Yes         12
Sales             No          20
                  Yes         12
dtype: int64

In [119]:
df.groupby(["Department", "Overtime"]).size()

Department        Overtime
Customer Support  No          144
                  Yes          58
Finance           No          137
                  Yes          41
HR                No           88
                  Yes          37
IT                No          210
                  Yes          88
Marketing         No          107
                  Yes          47
Operations        No          199
                  Yes          81
Sales             No          186
                  Yes          77
dtype: int64

In [120]:
((df[df["Attrition"] == "Yes"].groupby(["Department", "Overtime"]).size())/(df.groupby(["Department", "Overtime"]).size())*100).sort_values(ascending=False).reset_index().rename(columns={0: "Attrition_Rate"})


,Department,Overtime,Attrition_Rate
0,IT,Yes,18.181818
1,Sales,Yes,15.584416
2,Operations,Yes,14.814815
3,Customer Support,Yes,13.793103
4,Finance,Yes,12.195122
5,Finance,No,11.678832
6,Sales,No,10.752688
7,IT,No,9.047619
8,Operations,No,9.045226
9,Customer Support,No,9.027778


In [121]:
# Among employees who left the company, does overtime relate to their satisfaction level?

In [122]:
df[df["Attrition"]=="Yes"].groupby("Overtime")["Satisfaction_Score"].mean()

Overtime
No     2.926316
Yes    3.293103
Name: Satisfaction_Score, dtype: float64

In [123]:
# For employees who left the company, calculate the average satisfaction score for each Department and Overtime combination.

In [124]:
df[df["Attrition"]=="Yes"].groupby(["Overtime","Department"])["Satisfaction_Score"].mean()

Overtime  Department      
No        Customer Support    2.615385
          Finance             3.187500
          HR                  2.500000
          IT                  2.684211
          Marketing           2.285714
          Operations          3.333333
          Sales               3.050000
Yes       Customer Support    3.250000
          Finance             3.600000
          HR                  4.000000
          IT                  3.125000
          Marketing           4.000000
          Operations          3.416667
          Sales               3.000000
Name: Satisfaction_Score, dtype: float64

In [125]:
# For each department, compare the average satisfaction score of employees who stayed vs employees who left.

In [126]:
df.groupby(["Department","Attrition"])["Satisfaction_Score"].mean().unstack()

Attrition,No,Yes
Department,,
Customer Support,3.480663,2.857143
Finance,3.414013,3.285714
HR,3.608333,3.400000
IT,3.517110,2.885714
Marketing,3.544828,2.666667
Operations,3.500000,3.366667
Sales,3.489177,3.031250


In [127]:
# For each department, calculate the difference in average satisfaction between employees who stayed and employees who left

In [128]:
difference=df.groupby(["Department","Attrition"])["Satisfaction_Score"].mean().unstack()

In [129]:
attrition_satisfaction_diff=difference["Yes"]-difference["No"]
attrition_satisfaction_diff

Department
Customer Support   -0.623520
Finance            -0.128298
HR                 -0.208333
IT                 -0.631396
Marketing          -0.878161
Operations         -0.133333
Sales              -0.457927
dtype: float64

In [130]:
# Which department has the largest difference in satisfaction between employees who left and employees who stayed?
# largest absolute satisfaction gap.

In [131]:
attrition_satisfaction_diff=difference["Yes"]-difference["No"]
attrition_satisfaction_diff.abs().nlargest(1)

Department
Marketing    0.878161
dtype: float64

In [132]:
# For each department, calculate the average salary and average years of experience, then identify the department with the highest average salary.

In [133]:
dep_salary=df.groupby("Department")[["Salary","Years_Experience"]].mean()

In [134]:
dep_salary["Salary"].nlargest()

Department
Marketing     117392.857143
Finance       115277.528090
Operations    113538.214286
Sales         112100.760456
IT            107133.221477
Name: Salary, dtype: float64

In [135]:
dep_salary.loc[dep_salary["Salary"].nlargest(1).index] # find row

,Salary,Years_Experience
Department,,
Marketing,117392.857143,7.737013


In [136]:
# Calculate the average Satisfaction_Score for each Remote_Work category, and then sort the result from highest to lowest.

In [137]:
df.groupby("Remote_Work")["Satisfaction_Score"].mean().sort_values(ascending=False)

Remote_Work
Yes    3.491468
No     3.437637
Name: Satisfaction_Score, dtype: float64

In [138]:
# For each department, compare the average satisfaction score of remote and non-remote employees.

In [139]:
remote_satis=df.groupby(["Department","Remote_Work"])["Satisfaction_Score"].mean().unstack()

In [140]:
# For each department, calculate the difference between remote and non-remote employees' average satisfaction.

In [141]:
Difference_remote=remote_satis["Yes"]-remote_satis["No"]
Difference_remote

Department
Customer Support    0.203252
Finance             0.011513
HR                 -0.058962
IT                 -0.036140
Marketing           0.177769
Operations          0.022727
Sales               0.095065
dtype: float64

In [142]:
# Which department has the largest absolute difference in satisfaction between remote and non-remote employees?

In [143]:
Difference_remote=remote_satis["Yes"]-remote_satis["No"]
Difference_remote.abs().nlargest(1)

Department
Customer Support    0.203252
dtype: float64

# Transform:

In [144]:
# For every employee, calculate how their salary compares with the average salary of their own department.

In [145]:
Department_Avg_Salary=df.groupby("Department")["Salary"].transform("mean")
df["Department_Avg_Salary"]=Department_Avg_Salary

In [146]:
df["Salary_vs_Department_Avg"]=(df["Salary"]-df["Department_Avg_Salary"])
df["Salary_vs_Department_Avg"]

0       36561.785714
1       27522.471910
2       -5333.221477
3       22866.778523
4      -72133.221477
            ...     
1510     7422.471910
1511    59066.778523
1512   -60500.760456
1513    65722.471910
1514    20522.471910
Name: Salary_vs_Department_Avg, Length: 1500, dtype: float64

In [147]:
# For each employee, calculate their salary as a percentage above or below their department's average salary.

In [148]:
df["Salary_vs_Department_Avg_Pct"]=(df["Salary_vs_Department_Avg"]/df["Department_Avg_Salary"])*100

In [149]:
df.nlargest(5, "Salary_vs_Department_Avg_Pct")

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Overtime,Remote_Work,Projects_Completed,Training_Hours,Leave_Days,Promotion,Attrition,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct
1067,E10684,Employee_684,26.0,Female,Sales,Account Executive,1200000.0,1.8,2017-10-03,Karachi,...,Yes,Yes,12,25,5,No,No,112100.760456,1.087899e+06,970.465530
1144,E11182,Employee_1182,49.0,Male,Operations,Coordinator,920000.0,16.8,2017-05-01,Lahore,...,Yes,Yes,3,4,17,No,Yes,113538.214286,8.064618e+05,710.299868
490,E11086,Employee_1086,41.0,Male,Marketing,Marketing Manager,850000.0,9.0,2020-03-30,Islamabad,...,Yes,No,7,29,11,No,No,117392.857143,7.326071e+05,624.064497
927,E10131,Employee_131,29.0,Male,Operations,Coordinator,750000.0,4.3,2024-05-15,Rawalpindi,...,Yes,Yes,1,22,14,Yes,Yes,113538.214286,6.364618e+05,560.570544
1284,E11349,Employee_1349,41.0,Female,Finance,Finance Manager,680000.0,11.8,2022-06-16,Karachi,...,No,No,4,22,17,No,No,115277.528090,5.647225e+05,489.880796


In [150]:
# Calculate the attrition rate for each Department, separately for Overtime = Yes and Overtime = No, 
# and present the result as a table with Yes and No as columns.

In [151]:
overtime=df[df["Attrition"]=="Yes"].groupby(["Department","Overtime"]).size()

In [152]:
dep_over=df.groupby(["Department","Overtime"]).size()

In [153]:
Attrition_Rate=(overtime/dep_over)*100
Attrition_Rate.unstack()

Overtime,No,Yes
Department,,
Customer Support,9.027778,13.793103
Finance,11.678832,12.195122
HR,2.272727,8.108108
IT,9.047619,18.181818
Marketing,6.542056,4.255319
Operations,9.045226,14.814815
Sales,10.752688,15.584416


In [154]:
# For each department, calculate the average salary of employees who stayed vs employees who left.

In [155]:
salary_dep=df.groupby(["Department","Attrition"])["Salary"].mean().unstack()

In [156]:
avg_salary=salary_dep["Yes"]-salary_dep["No"]

In [157]:
avg_salary.abs().nlargest(1)

Department
HR    26692.5
dtype: float64

In [158]:
# Promotion analysis

In [159]:
# For each department, calculate the promotion rate separately for employees who work remotely and employees who don't.

In [160]:
Promoted_employees=df[df["Promotion"]=="Yes"].groupby(["Department","Remote_Work"]).size()

In [161]:
Total_employees=df.groupby(["Department", "Remote_Work"]).size()

In [162]:
Promotion_Rate=(Promoted_employees / Total_employees)*100
Promotion_Rate.unstack()

Remote_Work,No,Yes
Department,,
Customer Support,18.333333,14.634146
Finance,18.421053,17.187500
HR,12.500000,16.981132
IT,20.903955,18.181818
Marketing,21.495327,17.021277
Operations,24.431818,20.192308
Sales,16.216216,20.000000


In [163]:
# Count promoted employees for each Department + Remote_Work.
promoted=df[df["Promotion"]=="Yes"].groupby(["Department","Remote_Work"]).size()

In [164]:
total_employees=df.groupby(["Department","Remote_Work"]).size()

In [165]:
promotions_rate=(promoted/total_employees)*100
promotions_rate.unstack()

Remote_Work,No,Yes
Department,,
Customer Support,18.333333,14.634146
Finance,18.421053,17.187500
HR,12.500000,16.981132
IT,20.903955,18.181818
Marketing,21.495327,17.021277
Operations,24.431818,20.192308
Sales,16.216216,20.000000


In [166]:
# For each department, calculate the average satisfaction score separately for employees who work overtime and employees who don't.

In [167]:
df.groupby(["Department","Overtime"])["Satisfaction_Score"].mean().unstack()

Overtime,No,Yes
Department,,
Customer Support,3.381944,3.500000
Finance,3.401460,3.390244
HR,3.625000,3.540541
IT,3.471429,3.375000
Marketing,3.476636,3.531915
Operations,3.482412,3.493827
Sales,3.403226,3.506494


In [168]:
# For each department, compare the average salary of employees who were promoted with those who were not promoted, 
# and find the department with the largest salary difference.

In [169]:
salary_avg=df.groupby(["Department","Promotion"])["Salary"].mean().unstack()

In [170]:
dep_salary=salary_avg["Yes"]-salary_avg["No"]
dep_salary.nlargest(1)

Department
Operations    27180.787037
dtype: float64

In [171]:
dep_avg=df.groupby("Department")[["Performance_Score","Satisfaction_Score"]].mean()

In [172]:
dep_diff=dep_avg["Performance_Score"]-dep_avg["Satisfaction_Score"]
dep_diff.nlargest(1)

Department
Customer Support    0.00495
dtype: float64

In [173]:
# Find the average salary of employees in each department, but only for employees with more than 5 years of experience.
# Then find the department with the highest average salary.

In [174]:
salary_exp=df[df["Years_Experience"]>5].groupby("Department")["Salary"].mean()

In [175]:
salary_exp.nlargest(1)

Department
Finance    141225.210084
Name: Salary, dtype: float64

In [176]:
# Among employees who work overtime, find the average satisfaction score for each department, 
# then identify the department with the lowest average satisfaction.

In [177]:
satisfaction_overtime=df[df["Overtime"]=="Yes"].groupby("Department")["Satisfaction_Score"].mean()

In [178]:
satisfaction_overtime.nsmallest(1)

Department
IT    3.375
Name: Satisfaction_Score, dtype: float64

In [179]:
# For each department, calculate the percentage of employees who work remotely. 
# Then identify the department with the highest remote-work percentage.

In [180]:
remote_work=df[df["Remote_Work"]=="Yes"].groupby("Department").size()

In [181]:
remote_total=df.groupby("Department")["Remote_Work"].size()

In [182]:
percentage_remote=(remote_work/remote_total)*100

In [183]:
percentage_remote.nlargest(1)

Department
Sales    43.726236
dtype: float64

In [184]:
# For each department, calculate the average salary of employees with more than 5 years of experience, separately for remote and non-remote employees. 
# Then find the highest average salary among these groups.

In [185]:
remote_employee=df[df["Years_Experience"]>5].groupby(["Department","Remote_Work"])["Salary"].mean()

In [186]:
remote_employee.nlargest(1)

Department  Remote_Work
Operations  Yes            143672.727273
Name: Salary, dtype: float64

In [187]:
# Among employees who left the company, for each department 
# calculate the average years of experience separately for employees who work overtime and those who don't.
# Then find the group with the highest average experience.

In [188]:
avg_exp=df[df["Attrition"]=="Yes"].groupby(["Department","Overtime"])["Years_Experience"].mean()

In [189]:
avg_exp.nlargest(1)

Department  Overtime
Marketing   Yes         14.7
Name: Years_Experience, dtype: float64

In [190]:
# Among employees who have not left the company, for each department 
# calculate the average salary separately for employees who received a promotion and those who did not.
# Then find the group with the lowest average salary.

In [191]:
no_attrition=df[df["Attrition"]=="No"].groupby(["Department","Promotion"])["Salary"].mean()

In [192]:
no_attrition.nsmallest(1)

Department  Promotion
HR          No           100821.568627
Name: Salary, dtype: float64

In [193]:
# Among employees who work remotely, for each department 
# calculate the average satisfaction score separately for employees who work overtime and those who don't. 
# Then find the department/overtime group with the highest satisfaction score.

In [194]:
satisfaction_rate=df[df["Remote_Work"]=="Yes"].groupby(["Department","Overtime"])["Satisfaction_Score"].mean()

In [195]:
satisfaction_rate.nlargest(1)

Department        Overtime
Customer Support  Yes         3.772727
Name: Satisfaction_Score, dtype: float64

# introduce transform()

In [196]:
# For every employee, calculate how much their salary is above or below the average salary of their department.

# Then identify the 5 employees whose salary is furthest above their department's average.

In [197]:
dep_avg=df.groupby("Department")["Salary"].transform("mean")

In [198]:
df["difference_salary"]=df["Salary"]-dep_avg

In [199]:
df.nlargest(5,"difference_salary")

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Remote_Work,Projects_Completed,Training_Hours,Leave_Days,Promotion,Attrition,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct,difference_salary
1067,E10684,Employee_684,26.0,Female,Sales,Account Executive,1200000.0,1.8,2017-10-03,Karachi,...,Yes,12,25,5,No,No,112100.760456,1.087899e+06,970.465530,1.087899e+06
1144,E11182,Employee_1182,49.0,Male,Operations,Coordinator,920000.0,16.8,2017-05-01,Lahore,...,Yes,3,4,17,No,Yes,113538.214286,8.064618e+05,710.299868,8.064618e+05
490,E11086,Employee_1086,41.0,Male,Marketing,Marketing Manager,850000.0,9.0,2020-03-30,Islamabad,...,No,7,29,11,No,No,117392.857143,7.326071e+05,624.064497,7.326071e+05
927,E10131,Employee_131,29.0,Male,Operations,Coordinator,750000.0,4.3,2024-05-15,Rawalpindi,...,Yes,1,22,14,Yes,Yes,113538.214286,6.364618e+05,560.570544,6.364618e+05
1284,E11349,Employee_1349,41.0,Female,Finance,Finance Manager,680000.0,11.8,2022-06-16,Karachi,...,No,4,22,17,No,No,115277.528090,5.647225e+05,489.880796,5.647225e+05


In [200]:
# For each department, calculate the average salary of employees with more than 5 years of experience, 
# separately for promoted and non-promoted employees. 
# Then find the group with the largest average salary.

In [201]:
salary_avg=df[df["Years_Experience"]>5].groupby(["Department","Promotion"])["Salary"].mean()

In [202]:
salary_avg.nlargest(1)

Department  Promotion
Finance     No           144252.808989
Name: Salary, dtype: float64

In [203]:
# For each employee, calculate their department's average performance score,
# and then calculate how much their individual performance score differs from their department average.
# Finally, find the 5 employees furthest above their department average.

In [204]:
dep_performnce=df.groupby("Department")["Performance_Score"].transform("mean")

In [205]:
per_diff=df["Performance_Score"]-dep_performnce

In [206]:
per_diff.nlargest(5)

254    1.856
282    1.856
398    1.856
786    1.856
860    1.856
Name: Performance_Score, dtype: float64

In [207]:
# For employees who left the company, calculate each employee's difference from their department's average salary among employees who also left.

# Then find the 5 employees whose salary is furthest above that department-specific attrition-group average.

In [208]:
dep_employee_diff=df[df["Attrition"]=="Yes"].groupby("Department")["Salary"].transform("mean")

In [209]:
salary_diff = df["Salary"] - dep_employee_diff

In [210]:
salary_diff.nlargest(5)

1144    782913.333333
927     612913.333333
757     132714.285714
1069    127393.750000
889      86993.750000
Name: Salary, dtype: float64

#  pivot_table()

In [211]:
# Using the HR dataset, create a table showing the average salary for each Department, 
# split into Promoted (Yes) and Not Promoted (No) employees.

In [212]:
df.pivot_table(
    values="Salary",
    index="Department",
    columns="Promotion",
    aggfunc="mean"
)

Promotion,No,Yes
Department,,
Customer Support,99290.178571,126350.000000
Finance,112308.219178,128825.000000
HR,102196.261682,118994.444444
IT,103555.230126,121627.118644
Marketing,114852.032520,127474.193548
Operations,107325.462963,134506.250000
Sales,110422.685185,119812.765957


In [213]:
# Create a pivot table showing the average Satisfaction Score for each Department, split by Overtime, 
# but only for employees who have not left the company.

In [214]:
df[df["Attrition"]=="No"].pivot_table(
    values="Satisfaction_Score",
    index="Department",
    columns="Overtime",
    aggfunc="mean"
)

Overtime,No,Yes
Department,,
Customer Support,3.458015,3.540000
Finance,3.429752,3.361111
HR,3.651163,3.500000
IT,3.549738,3.430556
Marketing,3.560000,3.511111
Operations,3.497238,3.507246
Sales,3.445783,3.600000


In [215]:
# Create a pivot table showing both the average Salary and average Satisfaction Score for each Department, split by Promotion status.

In [216]:
salary_pivot=df.pivot_table(
    index="Department",
    columns="Promotion",
    values=["Salary","Satisfaction_Score"],
    aggfunc="mean"
)

In [217]:
salary_pivot

Salary                Satisfaction_Score          
Promotion                    No            Yes                 No       Yes
Department                                                                 
Customer Support   99290.178571  126350.000000           3.404762  3.470588
Finance           112308.219178  128825.000000           3.424658  3.281250
HR                102196.261682  118994.444444           3.560748  3.833333
IT                103555.230126  121627.118644           3.435146  3.474576
Marketing         114852.032520  127474.193548           3.528455  3.354839
Operations        107325.462963  134506.250000           3.481481  3.500000
Sales             110422.685185  119812.765957           3.402778  3.574468

# pivot + comparison

In [218]:
# Using the same pivot table, determine which Department has the largest difference in average Salary between promoted and non-promoted employees.

In [219]:
salary_dif=salary_pivot["Salary"]["Yes"]-salary_pivot["Salary"]["No"]

In [220]:
salary_dif.nlargest(1)

Department
Operations    27180.787037
dtype: float64

# pivot_table() + multiple aggregations

In [221]:
df.pivot_table(
    values=["Salary","Performance_Score","Name"],
    index="Department",
    columns="Remote_Work",
    aggfunc=({
        "Salary":["mean"],
        "Performance_Score":["mean"],
        "Name":["count"]
}))

Name      Performance_Score                   Salary  \
                 count                   mean                     mean   
Remote_Work         No  Yes                No       Yes             No   
Department                                                               
Customer Support   120   82          3.466667  3.353659  100285.833333   
Finance            114   64          3.280702  3.234375  114080.701754   
HR                  72   53          3.138889  3.150943  108825.000000   
IT                 177  121          3.384181  3.388430  108973.446328   
Marketing          107   47          3.168224  3.361702  117887.850467   
Operations         176  104          3.460227  3.442308  107774.147727   
Sales              148  115          3.385135  3.217391  108992.567568   

                                 
                                 
Remote_Work                 Yes  
Department                       
Customer Support  109053.048780  
Finance           117409.375000  
HR                 98896.226415  
IT                104441.322314  
Marketing         116265.957447  
Operations        123292.788462  
Sales             116100.869565

In [222]:
attri=df[df["Attrition"]=="Yes"].pivot_table(
    values=["Name"],
    index="Department",
    columns="Remote_Work",
    aggfunc="count"
)

In [223]:
total_emp = df.pivot_table(
    values="Name",
    index="Department",
    columns="Remote_Work",
    aggfunc="count"
)

In [224]:
attri_rate=(attri/total_emp)*100

In [225]:
attri_rate

Name           
Remote_Work              No        Yes
Department                            
Customer Support  11.666667   8.536585
Finance           11.403509  12.500000
HR                 4.166667   3.773585
IT                11.864407  11.570248
Marketing          7.476636   2.127660
Operations        10.227273  11.538462
Sales              8.108108  17.391304

In [226]:
df.pivot_table(
    index="Department",
    columns="Attrition",
    values=["Salary","Name"],
    aggfunc=({
        "Salary":["mean","median","max"],
        "Name":"count"
            }))

Name         Salary                                          \
                 count            max                     mean                  
Attrition           No Yes         No       Yes             No            Yes   
Department                                                                      
Customer Support   181  21   236400.0  141700.0  105998.066298   85285.714286   
Finance            157  21   680000.0  232100.0  117403.184713   99385.714286   
HR                 120   5   191600.0  200700.0  103547.500000  130240.000000   
IT                 263  35   239400.0  188400.0  105936.501901  116125.714286   
Marketing          145   9   850000.0  210600.0  116297.241379  135044.444444   
Operations         250  30   256600.0  920000.0  110712.400000  137086.666667   
Sales              231  32  1200000.0  239000.0  112169.264069  111606.250000   

                                      
                    median            
Attrition               No       Yes  
Department                            
Customer Support  103300.0   91000.0  
Finance           113300.0   98600.0  
HR                100250.0  140200.0  
IT                103000.0  108000.0  
Marketing         111400.0  168400.0  
Operations        107300.0   87600.0  
Sales             106400.0  114600.0

# Time-based analysis

In [227]:
df["Join_Date"].dt.year.value_counts().sort_index()

Join_Date
2017.0    166
2018.0    146
2019.0    190
2020.0    168
2021.0    183
2022.0    166
2023.0    142
2024.0    146
2025.0    171
Name: count, dtype: int64

In [228]:
join_year=df["Join_Date"].dt.year

In [229]:
df.groupby(join_year)["Salary"].mean()

Join_Date
2017.0    121912.951807
2018.0    113195.890411
2019.0    111867.894737
2020.0    110745.238095
2021.0    108432.786885
2022.0    105623.192771
2023.0    107927.464789
2024.0    111321.575342
2025.0    105632.163743
Name: Salary, dtype: float64

# time + filtering

In [230]:
# Among employees who joined in 2024, calculate the average salary separately for each department. 
# Then find the department with the highest average salary.

In [231]:
df["year_specific"]=df["Join_Date"].dt.year

In [232]:
df["year_specific"]

0       2024.0
1       2021.0
2       2017.0
3       2019.0
4       2023.0
         ...  
1510    2021.0
1511    2019.0
1512    2018.0
1513    2024.0
1514    2022.0
Name: year_specific, Length: 1500, dtype: float64

In [233]:
specifc_year=df[df["year_specific"]==2024].groupby("Department")["Salary"].mean()

In [234]:
specifc_year.nlargest(1)

Department
Operations    141193.75
Name: Salary, dtype: float64

# date feature engineering

In [235]:
# For each employee, calculate how many years they have been with the company as of January 1, 2026.

In [236]:
today=pd.Timestamp("2026-1-1")

In [237]:
difference_year=today-df["Join_Date"]

In [238]:
difference_year

0       620 days
1      1615 days
2      3128 days
3      2240 days
4      1081 days
          ...   
1510   1497 days
1511   2470 days
1512   2674 days
1513    570 days
1514   1123 days
Name: Join_Date, Length: 1500, dtype: timedelta64[us]

In [239]:
diff_days=difference_year.dt.days

In [240]:
diff_days

0        620.0
1       1615.0
2       3128.0
3       2240.0
4       1081.0
         ...  
1510    1497.0
1511    2470.0
1512    2674.0
1513     570.0
1514    1123.0
Name: Join_Date, Length: 1500, dtype: float64

In [241]:
years_worked=diff_days/365

In [242]:
years_worked

0       1.698630
1       4.424658
2       8.569863
3       6.136986
4       2.961644
          ...   
1510    4.101370
1511    6.767123
1512    7.326027
1513    1.561644
1514    3.076712
Name: Join_Date, Length: 1500, dtype: float64

In [243]:
# Using Join_Date, calculate the average Years_Experience of employees who joined in each year.

In [244]:
df["join_Year"]=df["Join_Date"].dt.year

In [245]:
df.groupby("join_Year")["Years_Experience"].mean()

join_Year
2017.0    7.519277
2018.0    7.763699
2019.0    7.673158
2020.0    7.534524
2021.0    7.276503
2022.0    6.763253
2023.0    7.255634
2024.0    6.957534
2025.0    7.073099
Name: Years_Experience, dtype: float64

# combine time + filtering + grouping

In [246]:
dep_salary=df[df["join_Year"]>=2023].groupby(["Department","join_Year"])["Salary"].mean()

In [247]:
dep_salary.nlargest(1)

Department  join_Year
Operations  2024.0       141193.75
Name: Salary, dtype: float64

# time + pivot_table()

In [248]:
df.pivot_table(
    index="join_Year",
    columns="Department",
    values="Salary",
    aggfunc="mean"
)

Department,Customer Support,Finance,HR,IT,Marketing,Operations,Sales
join_Year,,,,,,,
2017.0,108816.666667,86858.333333,106685.714286,112364.516129,119784.210526,138915.151515,143837.878788
2018.0,128800.000000,125842.105263,113288.235294,115228.125000,111690.000000,105155.172414,104088.888889
2019.0,106846.296296,90932.000000,98325.000000,125269.230769,117121.052632,116152.857143,113354.545455
2020.0,102270.000000,115605.555556,111200.000000,101900.000000,139578.947368,108706.060606,108513.333333
2021.0,115427.586207,136570.588235,84114.285714,93842.592593,128523.809524,101075.000000,106624.242424
2022.0,107277.586207,136963.636364,106491.666667,100027.272727,96345.454545,104809.523810,91455.555556
2023.0,91164.285714,114514.285714,100307.692308,107027.272727,119247.500000,106548.214286,110300.000000
2024.0,87723.809524,116032.000000,116500.000000,104044.642857,97523.076923,141193.750000,110532.608696
2025.0,87750.000000,109662.500000,102783.333333,104745.714286,108907.500000,111377.419355,113295.000000


# missing dates

In [249]:
missing_value=df["Join_Date"].isna().sum()

In [250]:
total_emp=df["Name"].count()

In [251]:
percentage_missing=(missing_value/total_emp)*100

In [252]:
percentage_missing

np.float64(1.4666666666666666)

# missing values by group

In [253]:
# For each Department, calculate the percentage of employees who have a missing Join_Date.
# Then find the department with the highest missing-date percentage.

In [254]:
dep_missing=df[df["Join_Date"].isna()].groupby("Department")["Name"].count()

In [255]:
total_employee=df.groupby("Department")["Name"].count()

In [256]:
percentage_em=(dep_missing/total_employee)*100

In [257]:
percentage_em.nlargest(1)

Department
HR    2.4
Name: Name, dtype: float64

# transform() + missing values

In [258]:
df.groupby("Department")["Join_Date"].transform("median")

0      2021-02-08 00:00:00
1      2021-10-25 12:00:00
2      2021-04-27 00:00:00
3      2021-04-27 00:00:00
4      2021-04-27 00:00:00
               ...        
1510   2021-10-25 12:00:00
1511   2021-04-27 00:00:00
1512   2021-02-08 00:00:00
1513   2021-10-25 12:00:00
1514   2021-10-25 12:00:00
Name: Join_Date, Length: 1500, dtype: datetime64[us]

In [259]:
df["Join_Date"]=df["Join_Date"].fillna(df.groupby("Department")["Join_Date"].transform("median"))

In [260]:
df["Join_Date"].isna().sum()

np.int64(0)

# date feature engineering

In [261]:
df["Join_Month"]=df["Join_Date"].dt.month_name()

In [262]:
df["Join_Month"]

0           April
1            July
2            June
3        November
4         January
          ...    
1510     November
1511        March
1512    September
1513         June
1514     December
Name: Join_Month, Length: 1500, dtype: str

In [263]:
no_emp=df.groupby("Join_Month")["Name"].count()

In [264]:
no_emp.nlargest(1)

Join_Month
August    137
Name: Name, dtype: int64

In [265]:
common_month=df.groupby(["Department","Join_Month"])["Name"].count()

In [266]:
common_month

Department        Join_Month
Customer Support  April         18
                  August        26
                  December      16
                  February      23
                  January       11
                                ..
Sales             March         27
                  May           19
                  November      18
                  October       21
                  September     20
Name: Name, Length: 84, dtype: int64

In [267]:
common_month.groupby(level="Department").nlargest(1)

Department        Department        Join_Month
Customer Support  Customer Support  August        26
Finance           Finance           August        22
HR                HR                December      14
IT                IT                April         29
Marketing         Marketing         July          17
Operations        Operations        May           32
Sales             Sales             February      29
Name: Name, dtype: int64

In [268]:
df[df["join_Year"]==2024].groupby("Department")["Salary"].mean()

Department
Customer Support     87723.809524
Finance             116032.000000
HR                  116500.000000
IT                  104044.642857
Marketing            97523.076923
Operations          141193.750000
Sales               110532.608696
Name: Salary, dtype: float64

In [269]:
df[df["join_Year"]>=2023].groupby("Department")["Years_Experience"].mean()

Department
Customer Support    5.155172
Finance             7.863492
HR                  7.283721
IT                  6.894118
Marketing           7.081132
Operations          7.415663
Sales               7.718919
Name: Years_Experience, dtype: float64

In [270]:
avg_salary=df[df["join_Year"]>=2023].groupby("Department")["Salary"].mean()

In [271]:
avg_salary.nlargest(1)

Department
Operations    118369.879518
Name: Salary, dtype: float64

In [272]:
df[(df["join_Year"]>=2023)&(df["Remote_Work"]=="Yes")].groupby("Department")["Satisfaction_Score"].mean()

Department
Customer Support    3.481481
Finance             3.304348
HR                  3.150000
IT                  3.486486
Marketing           3.266667
Operations          3.361111
Sales               3.586207
Name: Satisfaction_Score, dtype: float64

In [273]:
# comparison

In [274]:
satisfaction_no=df[df["join_Year"]>=2023].groupby(["Department","Remote_Work"])["Satisfaction_Score"].mean().unstack()

In [275]:
satisfaction_no

Remote_Work,No,Yes
Department,,
Customer Support,3.193548,3.481481
Finance,3.375000,3.304348
HR,3.391304,3.150000
IT,3.208333,3.486486
Marketing,3.289474,3.266667
Operations,3.425532,3.361111
Sales,3.533333,3.586207


In [276]:
# comparison + finding the largest gap

In [277]:
diff_sat=satisfaction_no["Yes"]-satisfaction_no["No"]

In [278]:
diff_sat.abs().nlargest(1)

Department
Customer Support    0.287933
dtype: float64

In [279]:
years_left=df[(df["join_Year"]>=2023)&(df["Attrition"]=="Yes")].groupby("Department")["Years_Experience"].mean()

In [280]:
years_left.nlargest(1)

Department
HR    9.5
Name: Years_Experience, dtype: float64

# date + percentage

In [281]:
at_rate=df[(df["join_Year"]>=2023)&(df["Attrition"]=="Yes")].groupby("Department").size()

In [282]:
total_m=df[df["join_Year"]>=2023].groupby("Department").size()

In [283]:
Attrition_Rate=(at_rate/total_m)*100

In [284]:
Attrition_Rate

Department
Customer Support    13.793103
Finance             12.698413
HR                   4.651163
IT                  12.941176
Marketing            5.660377
Operations          12.048193
Sales                8.108108
dtype: float64

In [285]:
dep_att=df[(df["join_Year"]>=2023)&(df["Attrition"]=="Yes")].groupby(["Department","Remote_Work"]).size()

In [286]:
dep_remote=df[df["join_Year"]>=2023].groupby(["Department","Remote_Work"]).size()

In [287]:
attrition_rate=(dep_att/dep_remote)*100

In [288]:
attrition_rate.nlargest(1)

Department  Remote_Work
Operations  Yes            16.666667
dtype: float64

# Combining DataFrames

In [289]:
# Dataset 2:

In [290]:
performance_df = pd.DataFrame({
    "Employee_ID": [1001, 1002, 1003, 1004, 1005],
    "Projects_Completed": [8, 12, 5, 10, 7],
    "Training_Hours": [20, 35, 15, 28, 22]
})

# merge()

In [291]:
df["Employee_ID"]=df["Employee_ID"].astype(str)
performance_df["Employee_ID"]=performance_df["Employee_ID"].astype(str)

In [292]:
df.merge(performance_df, on="Employee_ID")

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Attrition,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month,Projects_Completed_y,Training_Hours_y


# Merge error → check the key columns and their dtypes first.

In [293]:
# if the key is different:but same values:we have to use both keys:

In [294]:
# .merge() : we use merge on comonon keys

In [295]:
# Two tables + common key → merge

In [296]:
# We only want to add Projects_Completed from performance_df, not Training_Hours.

In [297]:
df.merge(performance_df[["Projects_Completed","Employee_ID"]],on="Employee_ID")

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Promotion,Attrition,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month,Projects_Completed_y


# concat()

In [298]:
# stacking rows from DataFrames with the same kind of columns, use: concat()

In [299]:
# all_employees = pd.concat([jan_employees, feb_employees])

In [300]:
# concat() with columns

# Real project problem

Common key → merge()
Stack similar tables → concat()

# real-world merge problems

In [301]:
import pandas as pd

employees = pd.DataFrame({
    "Employee_ID": [101, 102, 103, 104],
    "Name": ["Ali", "Sara", "Ahmed", "Ayesha"],
    "Department": ["IT", "HR", "Sales", "Finance"]
})

salary = pd.DataFrame({
    "Employee_ID": [101, 102, 103, 104],
    "Salary": [80000, 75000, 90000, 70000]
})

In [302]:
employees.merge(salary,on="Employee_ID")

,Employee_ID,Name,Department,Salary
0,101,Ali,IT,80000
1,102,Sara,HR,75000
2,103,Ahmed,Sales,90000
3,104,Ayesha,Finance,70000


In [303]:
import pandas as pd

employees = pd.DataFrame({
    "Employee_ID": [101, 102, 103, 104],
    "Name": ["Ali", "Sara", "Ahmed", "Ayesha"],
    "Department": ["IT", "HR", "Sales", "Finance"]
})

salary = pd.DataFrame({
    "Employee_ID": [101, 102, 104, 106],
    "Salary": [80000, 75000, 70000, 95000]
})

In [304]:
employees.merge(salary,on="Employee_ID") # on= works on inner key,takes only matching keys

,Employee_ID,Name,Department,Salary
0,101,Ali,IT,80000
1,102,Sara,HR,75000
2,104,Ayesha,Finance,70000


In [305]:
# left

In [306]:
employees.merge(salary, on="Employee_ID",how="left") #(left,right)= (left=takes keys from left side ,employees)

,Employee_ID,Name,Department,Salary
0,101,Ali,IT,80000.0
1,102,Sara,HR,75000.0
2,103,Ahmed,Sales,NaN
3,104,Ayesha,Finance,70000.0


In [307]:
employees.merge(salary, on="Employee_ID",how="right")   # right= takes keys from right side=salary

,Employee_ID,Name,Department,Salary
0,101,Ali,IT,80000
1,102,Sara,HR,75000
2,104,Ayesha,Finance,70000
3,106,NaN,NaN,95000


In [308]:
employees.merge(salary, on="Employee_ID",how="outer")   # outer= takes keys from both

,Employee_ID,Name,Department,Salary
0,101,Ali,IT,80000.0
1,102,Sara,HR,75000.0
2,103,Ahmed,Sales,NaN
3,104,Ayesha,Finance,70000.0
4,106,NaN,NaN,95000.0


In [309]:
employees.merge(salary, on="Employee_ID",how="inner")  # inner= takes matching keys

,Employee_ID,Name,Department,Salary
0,101,Ali,IT,80000
1,102,Sara,HR,75000
2,104,Ayesha,Finance,70000


In [310]:
employees = pd.DataFrame({
    "Employee_ID": [101, 102, 103],
    "Name": ["Ali", "Sara", "Ahmed"]
})

salary = pd.DataFrame({
    "Emp_ID": [101, 102, 103],
    "Salary": [80000, 75000, 90000]
})

In [311]:
employees.merge(salary,left_on="Employee_ID",right_on="Emp_ID")

,Employee_ID,Name,Emp_ID,Salary
0,101,Ali,101,80000
1,102,Sara,102,75000
2,103,Ahmed,103,90000


In [312]:
# Duplicate keys

In [313]:
employees = pd.DataFrame({
    "Employee_ID": [101, 102, 103],
    "Name": ["Ali", "Sara", "Ahmed"]
})

performance = pd.DataFrame({
    "Employee_ID": [101, 101, 102],
    "Projects": [8, 10, 12]
})

In [314]:
employees.merge(performance,on="Employee_ID",how="left")

,Employee_ID,Name,Projects
0,101,Ali,8.0
1,101,Ali,10.0
2,102,Sara,12.0
3,103,Ahmed,NaN


In [315]:
# Rows created for a key = left occurrences × right occurrences

In [316]:
employees.merge(performance,on="Employee_ID",how="right")

,Employee_ID,Name,Projects
0,101,Ali,8
1,101,Ali,10
2,102,Sara,12


In [317]:
employees.merge(performance,on="Employee_ID",how="outer")

,Employee_ID,Name,Projects
0,101,Ali,8.0
1,101,Ali,10.0
2,102,Sara,12.0
3,103,Ahmed,NaN


In [318]:
employees.merge(performance,on="Employee_ID",how="inner")

,Employee_ID,Name,Projects
0,101,Ali,8
1,101,Ali,10
2,102,Sara,12


In [319]:
employees = pd.DataFrame({
    "Employee_ID": [101, 102, 103],
    "Name": ["Ali", "Sara", "Ahmed"]
})

performance = pd.DataFrame({
    "Employee_ID": [101, 101, 102, 102],
    "Projects": [5, 8, 10, 12]
})

In [320]:
employees.merge(
    performance,
    on="Employee_ID",
    how="left"
)

,Employee_ID,Name,Projects
0,101,Ali,5.0
1,101,Ali,8.0
2,102,Sara,10.0
3,102,Sara,12.0
4,103,Ahmed,NaN


In [321]:
employees.merge(
    performance,
    on="Employee_ID",
    how="right"
)

,Employee_ID,Name,Projects
0,101,Ali,5
1,101,Ali,8
2,102,Sara,10
3,102,Sara,12


In [322]:
employees.merge(
    performance,
    on="Employee_ID",
    how="outer"
)

,Employee_ID,Name,Projects
0,101,Ali,5.0
1,101,Ali,8.0
2,102,Sara,10.0
3,102,Sara,12.0
4,103,Ahmed,NaN


In [323]:
employees.merge(
    performance,
    on="Employee_ID",
    how="inner"
)

,Employee_ID,Name,Projects
0,101,Ali,5
1,101,Ali,8
2,102,Sara,10
3,102,Sara,12


In [324]:
# concat()

# merge() → match/combine based on a key
# concat() → stack DataFrames together

In [325]:
jan = pd.DataFrame({
    "Employee_ID": [101, 102, 103],
    "Name": ["Ali", "Sara", "Ahmed"],
    "Department": ["IT", "HR", "Sales"]
})

feb = pd.DataFrame({
    "Employee_ID": [104, 105, 106],
    "Name": ["Ayesha", "Usman", "Hina"],
    "Department": ["Finance", "IT", "Marketing"]
})

In [326]:
pd.concat([jan,feb])

,Employee_ID,Name,Department
0,101,Ali,IT
1,102,Sara,HR
2,103,Ahmed,Sales
0,104,Ayesha,Finance
1,105,Usman,IT
2,106,Hina,Marketing


In [327]:
pd.concat([jan,feb],axis=0, ignore_index=True)      # row vise

,Employee_ID,Name,Department
0,101,Ali,IT
1,102,Sara,HR
2,103,Ahmed,Sales
3,104,Ayesha,Finance
4,105,Usman,IT
5,106,Hina,Marketing


In [328]:
pd.concat([jan,feb],axis=1)

,Employee_ID,Name,Department,Employee_ID,Name,Department
0,101,Ali,IT,104,Ayesha,Finance
1,102,Sara,HR,105,Usman,IT
2,103,Ahmed,Sales,106,Hina,Marketing


In [329]:
jan = pd.DataFrame({
    "Employee_ID": [101, 102, 103],
    "Name": ["Ali", "Sara", "Ahmed"]
})

salary = pd.DataFrame({
    "Salary": [80000, 75000, 90000],
    "Bonus": [5000, 4000, 6000]
})

In [330]:
pd.concat([jan,salary],axis=1)   # column vise

,Employee_ID,Name,Salary,Bonus
0,101,Ali,80000,5000
1,102,Sara,75000,4000
2,103,Ahmed,90000,6000


In [331]:
df1 = pd.DataFrame({
    "Name": ["Ali", "Sara"],
    "Salary": [80000, 90000]
})

df2 = pd.DataFrame({
    "Name": ["Ahmed", "Usman"],
    "Department": ["IT", "HR"]
})

result = pd.concat([df1, df2], ignore_index=True)

In [332]:
result

,Name,Salary,Department
0,Ali,80000.0,NaN
1,Sara,90000.0,NaN
2,Ahmed,NaN,IT
3,Usman,NaN,HR


In [333]:
jan = pd.DataFrame({
    "Order_ID": ["J001", "J002", "J003"],
    "Product": ["Laptop", "Phone", "Mouse"],
    "Sales": [120000, 80000, 5000]
})

feb = pd.DataFrame({
    "Order_ID": ["F001", "F002"],
    "Product": ["Keyboard", "Monitor"],
    "Sales": [7000, 45000]
})

mar = pd.DataFrame({
    "Order_ID": ["M001", "M002", "M003"],
    "Product": ["Laptop", "Phone", "Headset"],
    "Sales": [125000, 85000, 12000]
})

In [334]:
# combine jan, feb, and mar into one DataFrame called q1_sales.

In [335]:
q1_sales=pd.concat([jan,feb,mar],ignore_index=True)

In [336]:
q1_sales

,Order_ID,Product,Sales
0,J001,Laptop,120000
1,J002,Phone,80000
2,J003,Mouse,5000
3,F001,Keyboard,7000
4,F002,Monitor,45000
5,M001,Laptop,125000
6,M002,Phone,85000
7,M003,Headset,12000


# concat() uses join="outer", so it keeps the union of all columns.

In [337]:
online = pd.DataFrame({
    "Order_ID": ["O1", "O2"],
    "Sales": [5000, 7000]
})

store = pd.DataFrame({
    "Order_ID": ["S1", "S2"],
    "Sales": [6000, 8000]
})

In [338]:
# Create one table containing all online and store orders

In [339]:
pd.concat([online,store],ignore_index=True)

,Order_ID,Sales
0,O1,5000
1,O2,7000
2,S1,6000
3,S2,8000


In [340]:
employees = pd.DataFrame({
    "Employee_ID": [101, 102, 103],
    "Name": ["Ali", "Sara", "Ahmed"],
    "Department": ["IT", "HR", "Finance"]
})

performance = pd.DataFrame({
    "Employee_ID": [101, 102, 103],
    "Performance_Score": [85, 92, 78]
})

In [341]:
# one table containing each employee's department AND their performance score.

In [342]:
employees.merge(performance[["Employee_ID","Performance_Score"]],on="Employee_ID")

,Employee_ID,Name,Department,Performance_Score
0,101,Ali,IT,85
1,102,Sara,HR,92
2,103,Ahmed,Finance,78


In [343]:
jan_perf = pd.DataFrame({
    "Employee_ID": [101, 102],
    "Performance_Score": [85, 90]
})

feb_perf = pd.DataFrame({
    "Employee_ID": [103, 104],
    "Performance_Score": [78, 88]
})

In [344]:
# Create all_performance by combining jan_perf and feb_perf.

In [345]:
all_performance=pd.concat([jan_perf,feb_perf],ignore_index=True)

In [346]:
all_performance

,Employee_ID,Performance_Score
0,101,85
1,102,90
2,103,78
3,104,88


In [347]:
final_employee_data=employees.merge(all_performance[["Employee_ID","Performance_Score"]],on="Employee_ID")

In [348]:
final_employee_data

,Employee_ID,Name,Department,Performance_Score
0,101,Ali,IT,85
1,102,Sara,HR,90
2,103,Ahmed,Finance,78


# pivot_table()

In [349]:
sales = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR", "Finance", "Finance"],
    "Gender": ["Male", "Female", "Female", "Male", "Male", "Female"],
    "Salary": [80000, 90000, 60000, 65000, 95000, 85000]
})

In [350]:
# What is the average salary for each Department and Gender?

In [351]:
sales.pivot_table(
    index="Department",
    columns="Gender",
    values="Salary",
    aggfunc="mean"
)

Gender,Female,Male
Department,,
Finance,85000.0,95000.0
HR,60000.0,65000.0
IT,90000.0,80000.0


In [352]:
# How many employees of each Gender are in each Department

In [353]:
sales.pivot_table(
    index="Gender",
    columns="Department",
    values="Salary",
    aggfunc="count"
)


Department,Finance,HR,IT
Gender,,,
Female,1,1,1
Male,1,1,1


In [354]:
# What is the total salary paid by each Department and Gender?"

In [355]:
sales.pivot_table(
    index="Department",
    columns="Gender",
    values="Salary",
    aggfunc="sum"
)

Gender,Female,Male
Department,,
Finance,85000,95000
HR,60000,65000
IT,90000,80000


In [356]:
# "Show both the average salary AND the total salary for each Department and Gender."

In [357]:
sales.pivot_table(
    index="Department",
    columns="Gender",
    values="Salary",
    aggfunc=["sum","mean"]
)

sum            mean         
Gender     Female   Male   Female     Male
Department                                
Finance     85000  95000  85000.0  95000.0
HR          60000  65000  60000.0  65000.0
IT          90000  80000  90000.0  80000.0

In [358]:
# "For each department, show the average salary, highest salary, lowest salary, and number of employees."

In [359]:
sales.pivot_table(
    index="Department",
    values="Salary",
    aggfunc=["mean","max","min","count"]
)

,mean,max,min,count
,Salary,Salary,Salary,Salary
Department,,,,
Finance,90000.0,95000,85000,2
HR,62500.0,65000,60000,2
IT,85000.0,90000,80000,2


In [360]:
# "For each department, show the number of employees and the total salary, separated by gender."

In [361]:
sales.pivot_table(
    index="Department",
    columns="Gender",
    values="Salary",
    aggfunc=["count","sum"]
)

count         sum       
Gender     Female Male Female   Male
Department                          
Finance         1    1  85000  95000
HR              1    1  60000  65000
IT              1    1  90000  80000

In [362]:
# pivot_table() with a calculated rate

In [363]:
hr = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR", "Finance", "Finance"],
    "Employee_ID": [1, 2, 3, 4, 5, 6],
    "Attrition": ["Yes", "No", "Yes", "No", "No", "No"]
})

In [364]:
hr["left"]=np.where(hr["Attrition"]=="Yes",1,0)

In [365]:
hr

,Department,Employee_ID,Attrition,left
0,IT,1,Yes,1
1,IT,2,No,0
2,HR,3,Yes,1
3,HR,4,No,0
4,Finance,5,No,0
5,Finance,6,No,0


In [366]:
summry=hr.pivot_table(
    index="Department",
    values=["left","Employee_ID"],
    aggfunc=({
        "Employee_ID":"count",
        "left":"sum"
    })
            )

In [367]:
summry

,Employee_ID,left
Department,,
Finance,2,0
HR,2,1
IT,2,1


In [368]:
summry_attrotion=(summry["left"]/summry["Employee_ID"])*100

In [369]:
summry_attrotion

Department
Finance     0.0
HR         50.0
IT         50.0
dtype: float64

In [370]:
# "For each department and gender, show the average Performance_Score and average Satisfaction_Score."

In [371]:
hr=pd.read_csv("employee_hr_analytics.csv")

In [372]:
df

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Leave_Days,Promotion,Attrition,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month
0,E10674,Employee_674,43.0,Male,Operations,Coordinator,150100.0,13.9,2024-04-21,Peshawar,...,8,Yes,No,113538.214286,36561.785714,32.202185,36561.785714,2024.0,2024.0,April
1,E10204,Employee_204,34.0,Female,Finance,Finance Manager,142800.0,9.0,2021-07-31,Rawalpindi,...,9,No,Yes,115277.528090,27522.471910,23.874967,27522.471910,2021.0,2021.0,July
2,E10308,Employee_308,41.0,Male,IT,Software Engineer,101800.0,10.1,2017-06-09,Peshawar,...,12,Yes,No,107133.221477,-5333.221477,-4.978121,-5333.221477,2017.0,2017.0,June
3,E11333,Employee_1333,35.0,Male,IT,IT Support Specialist,130000.0,8.9,2019-11-14,Islamabad,...,16,No,No,107133.221477,22866.778523,21.344246,22866.778523,2019.0,2019.0,November
4,E10492,Employee_492,22.0,Female,IT,Data Analyst,35000.0,0.0,2023-01-16,Islamabad,...,12,No,No,107133.221477,-72133.221477,-67.330395,-72133.221477,2023.0,2023.0,January
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1510,E11093,Employee_1093,35.0,Female,Finance,Financial Analyst,122700.0,7.0,2021-11-26,Peshawar,...,15,No,No,115277.528090,7422.471910,6.438785,7422.471910,2021.0,2021.0,November
1511,E11193,Employee_1193,41.0,Male,IT,Software Engineer,166200.0,10.3,2019-03-29,Rawalpindi,...,17,No,No,107133.221477,59066.778523,55.133952,59066.778523,2019.0,2019.0,March
1512,E11210,Employee_1210,24.0,Female,Sales,Account Executive,51600.0,1.4,2018-09-06,Lahore,...,9,No,No,112100.760456,-60500.760456,-53.969982,-60500.760456,2018.0,2018.0,September
1513,E11060,Employee_1060,50.0,Female,Finance,Finance Manager,181000.0,14.6,2024-06-10,Multan,...,14,No,No,115277.528090,65722.471910,57.012388,65722.471910,2024.0,2024.0,June


In [373]:
df.pivot_table(
    index="Department",
    columns="Gender",
    values=["Performance_Score","Satisfaction_Score"],
    aggfunc="mean"
)

Performance_Score           Satisfaction_Score          
Gender                      Female      Male             Female      Male
Department                                                               
Customer Support          3.377358  3.468750           3.349057  3.489583
Finance                   3.181818  3.344444           3.318182  3.477778
HR                        3.137931  3.149254           3.551724  3.641791
IT                        3.442029  3.337500           3.413043  3.468750
Marketing                 3.153846  3.264706           3.500000  3.490196
Operations                3.487603  3.427673           3.330579  3.603774
Sales                     3.360656  3.269504           3.327869  3.524823

In [374]:
# For each department, how many employees are working overtime vs not working overtime?

In [375]:
overtime_work=np.where(df["Overtime"]=="Yes",1,0)

In [376]:
df.pivot_table(
    index="Department",
    columns="Overtime",
    values="Name",
    aggfunc="count"
)

Overtime,No,Yes
Department,,
Customer Support,144,58
Finance,137,41
HR,88,37
IT,210,88
Marketing,107,47
Operations,199,81
Sales,186,77


In [377]:
import pandas as pd

df1 = pd.DataFrame({
    "Employee_ID": [1, 2, 3, 4, 5, 6],
    "Department": ["IT", "IT", "IT", "HR", "HR", "Finance"],
    "Attrition": ["Yes", "No", "No", "Yes", "No", "Yes"]
})

df["Left"] = (df["Attrition"] == "Yes").astype(int)

In [378]:
attri_left=df.groupby("Department")["Left"].sum()

In [379]:
total_employee=df.groupby("Department")["Employee_ID"].count()

In [380]:
attri_rate=(attri_left/total_employee)*100

In [381]:
attri_rate

Department
Customer Support    10.396040
Finance             11.797753
HR                   4.000000
IT                  11.744966
Marketing            5.844156
Operations          10.714286
Sales               12.167300
dtype: float64

In [382]:
sum1=df[df["Attrition"]=="Yes"].groupby("Department")["Employee_ID"].count()

In [383]:
sum2=df.groupby("Department")["Employee_ID"].count()

In [384]:
attri_rate=(sum1/sum2)*100

In [385]:
attri_rate.nlargest(1)

Department
Sales    12.1673
Name: Employee_ID, dtype: float64

# pd.crosstab()

In [386]:
df2 = pd.DataFrame({
    "Department": ["IT", "IT", "IT", "HR", "HR", "Finance"],
    "Gender": ["Male", "Female", "Male", "Female", "Male", "Female"],
    "Overtime": ["Yes", "No", "Yes", "No", "Yes", "Yes"]
})

In [387]:
pd.crosstab(df2["Department"],df2["Overtime"])

Overtime,No,Yes
Department,,
Finance,0,1
HR,1,1
IT,1,2


In [388]:
pd.crosstab(df2["Department"],df2["Gender"])

Gender,Female,Male
Department,,
Finance,1,0
HR,1,1
IT,1,2


In [389]:
pd.crosstab(df2["Department"],df2["Gender"],normalize="index")*100

Gender,Female,Male
Department,,
Finance,100.000000,0.000000
HR,50.000000,50.000000
IT,33.333333,66.666667


In [390]:
pd.crosstab(df2["Department"],df2["Gender"],normalize="columns")*100

Gender,Female,Male
Department,,
Finance,33.333333,0.000000
HR,33.333333,33.333333
IT,33.333333,66.666667


In [391]:
pd.crosstab(df2["Department"],df2["Gender"],normalize="columns")

Gender,Female,Male
Department,,
Finance,0.333333,0.000000
HR,0.333333,0.333333
IT,0.333333,0.666667


# melt()

In [392]:
df3 = pd.DataFrame({
    "Employee": ["Ali", "Sara"],
    "Salary": [80000, 90000],
    "Bonus": [5000, 7000]
})

In [393]:
pd.melt(
    df3,
    id_vars="Employee",  # identical variable
    var_name="variable",
    value_name="value"
)

,Employee,variable,value
0,Ali,Salary,80000
1,Sara,Salary,90000
2,Ali,Bonus,5000
3,Sara,Bonus,7000


In [394]:
df4 = pd.DataFrame({
    "Employee": ["Ali", "Sara", "Ahmed"],
    "Department": ["IT", "HR", "Finance"],
    "Jan_Sales": [100, 150, 120],
    "Feb_Sales": [130, 160, 140],
    "Mar_Sales": [125, 170, 155]
})

In [395]:
pd.melt(
    df4,
    id_vars=["Employee","Department"],
    var_name="Months",
    value_name="sales"
)

,Employee,Department,Months,sales
0,Ali,IT,Jan_Sales,100
1,Sara,HR,Jan_Sales,150
2,Ahmed,Finance,Jan_Sales,120
3,Ali,IT,Feb_Sales,130
4,Sara,HR,Feb_Sales,160
5,Ahmed,Finance,Feb_Sales,140
6,Ali,IT,Mar_Sales,125
7,Sara,HR,Mar_Sales,170
8,Ahmed,Finance,Mar_Sales,155


In [396]:
df4.reset_index()

,index,Employee,Department,Jan_Sales,Feb_Sales,Mar_Sales
0,0,Ali,IT,100,130,125
1,1,Sara,HR,150,160,170
2,2,Ahmed,Finance,120,140,155


In [397]:
# transform()

In [398]:
df12 = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR"],
    "Salary": [80000, 90000, 60000, 70000]
})

In [399]:
# “For every employee, show their salary and the average salary of their department.”

In [400]:
df12.groupby("Department")["Salary"].transform("mean")

0    85000.0
1    85000.0
2    65000.0
3    65000.0
Name: Salary, dtype: float64

In [401]:
# rank()

In [402]:
df11 = pd.DataFrame({
    "Department": ["IT", "IT", "IT", "HR", "HR"],
    "Employee": ["Ali", "Sara", "Ahmed", "Usman", "Ayesha"],
    "Salary": [80000, 90000, 75000, 60000, 70000]
})

In [403]:
#“Rank employees by salary within their own department.”
df11.groupby("Department")["Salary"].rank(ascending=False)

0    2.0
1    1.0
2    3.0
3    2.0
4    1.0
Name: Salary, dtype: float64

In [404]:
# tie rank()

In [405]:
df11.groupby("Department")["Salary"].rank(ascending=False,method="min")

0    2.0
1    1.0
2    3.0
3    2.0
4    1.0
Name: Salary, dtype: float64

In [406]:
# rolling() — moving calculations

In [407]:
# df["Sales"].rolling(3).mean()

In [408]:
# date-based rolling calculations.

In [409]:
df22 = pd.DataFrame({
    "Date": pd.to_datetime([
        "2026-01-01",
        "2026-01-02",
        "2026-01-04",
        "2026-01-05",
        "2026-01-07"
    ]),
    "Sales": [100, 120, 150, 130, 180]
})

In [410]:
df22["Sales"].rolling(3).mean()

0           NaN
1           NaN
2    123.333333
3    133.333333
4    153.333333
Name: Sales, dtype: float64

# expanding()

In [411]:
# Start from the first observation and keep expanding the window.

# Advanced Date/Time — resample()

In [412]:
# groupby():is mainly for categories:
# resample(): is for time periods:

In [413]:
# “How many orders were placed in each quarter?”
# df["Order_Date"].resample("QE").count()

# pd.Grouper

In [414]:
# combine time grouping + another category.

In [415]:
# df.groupby(["Department",pd.Grouper(key="Order_Date".freq="ME")])["Sales"].sum()

In [416]:
df23 = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR", "IT"],
    "Order_Date": pd.to_datetime([
        "2026-01-05",
        "2026-01-20",
        "2026-01-10",
        "2026-02-15",
        "2026-02-20"
    ]),
    "Sales": [100, 200, 150, 300, 250]
})

In [417]:
# “What are the total sales for each department in each month?”   /(total-sales,dep,each-month)?

In [418]:
df23.groupby(["Department",pd.Grouper(key="Order_Date",freq="ME")])["Sales"].sum()

Department  Order_Date
HR          2026-01-31    150
            2026-02-28    300
IT          2026-01-31    300
            2026-02-28    250
Name: Sales, dtype: int64

# “What is X for each category, over time?”
# groupby([category, pd.Grouper(date_column, frequency)])

In [419]:
# Pandas performance and vectorization

In [420]:
df23.memory_usage(deep=True).sum()

np.int64(467)

In [421]:
df

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Promotion,Attrition,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month,Left
0,E10674,Employee_674,43.0,Male,Operations,Coordinator,150100.0,13.9,2024-04-21,Peshawar,...,Yes,No,113538.214286,36561.785714,32.202185,36561.785714,2024.0,2024.0,April,0
1,E10204,Employee_204,34.0,Female,Finance,Finance Manager,142800.0,9.0,2021-07-31,Rawalpindi,...,No,Yes,115277.528090,27522.471910,23.874967,27522.471910,2021.0,2021.0,July,1
2,E10308,Employee_308,41.0,Male,IT,Software Engineer,101800.0,10.1,2017-06-09,Peshawar,...,Yes,No,107133.221477,-5333.221477,-4.978121,-5333.221477,2017.0,2017.0,June,0
3,E11333,Employee_1333,35.0,Male,IT,IT Support Specialist,130000.0,8.9,2019-11-14,Islamabad,...,No,No,107133.221477,22866.778523,21.344246,22866.778523,2019.0,2019.0,November,0
4,E10492,Employee_492,22.0,Female,IT,Data Analyst,35000.0,0.0,2023-01-16,Islamabad,...,No,No,107133.221477,-72133.221477,-67.330395,-72133.221477,2023.0,2023.0,January,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1510,E11093,Employee_1093,35.0,Female,Finance,Financial Analyst,122700.0,7.0,2021-11-26,Peshawar,...,No,No,115277.528090,7422.471910,6.438785,7422.471910,2021.0,2021.0,November,0
1511,E11193,Employee_1193,41.0,Male,IT,Software Engineer,166200.0,10.3,2019-03-29,Rawalpindi,...,No,No,107133.221477,59066.778523,55.133952,59066.778523,2019.0,2019.0,March,0
1512,E11210,Employee_1210,24.0,Female,Sales,Account Executive,51600.0,1.4,2018-09-06,Lahore,...,No,No,112100.760456,-60500.760456,-53.969982,-60500.760456,2018.0,2018.0,September,0
1513,E11060,Employee_1060,50.0,Female,Finance,Finance Manager,181000.0,14.6,2024-06-10,Multan,...,No,No,115277.528090,65722.471910,57.012388,65722.471910,2024.0,2024.0,June,0


In [422]:
df["Dept_Avg_Salary"]=df.groupby("Department")["Salary"].transform("mean")

In [423]:
# Which employees earn above their department's average salary?

In [424]:
df[df["Salary"]>df["Dept_Avg_Salary"]]

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Attrition,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month,Left,Dept_Avg_Salary
0,E10674,Employee_674,43.0,Male,Operations,Coordinator,150100.0,13.9,2024-04-21,Peshawar,...,No,113538.214286,36561.785714,32.202185,36561.785714,2024.0,2024.0,April,0,113538.214286
1,E10204,Employee_204,34.0,Female,Finance,Finance Manager,142800.0,9.0,2021-07-31,Rawalpindi,...,Yes,115277.528090,27522.471910,23.874967,27522.471910,2021.0,2021.0,July,1,115277.528090
3,E11333,Employee_1333,35.0,Male,IT,IT Support Specialist,130000.0,8.9,2019-11-14,Islamabad,...,No,107133.221477,22866.778523,21.344246,22866.778523,2019.0,2019.0,November,0,107133.221477
6,E10723,Employee_723,36.0,Male,IT,Data Analyst,111100.0,6.5,2022-07-23,Lahore,...,No,107133.221477,3966.778523,3.702660,3966.778523,2022.0,2022.0,July,0,107133.221477
12,E10390,Employee_390,34.0,Female,Customer Support,Support Representative,146500.0,9.3,2020-02-16,Peshawar,...,No,103844.801980,42655.198020,41.075911,42655.198020,2020.0,2020.0,February,0,103844.801980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1509,E11222,Employee_1222,44.0,Male,Operations,Coordinator,193900.0,15.9,2020-07-14,Islamabad,...,No,113538.214286,80361.785714,70.779505,80361.785714,2020.0,2020.0,July,0,113538.214286
1510,E11093,Employee_1093,35.0,Female,Finance,Financial Analyst,122700.0,7.0,2021-11-26,Peshawar,...,No,115277.528090,7422.471910,6.438785,7422.471910,2021.0,2021.0,November,0,115277.528090
1511,E11193,Employee_1193,41.0,Male,IT,Software Engineer,166200.0,10.3,2019-03-29,Rawalpindi,...,No,107133.221477,59066.778523,55.133952,59066.778523,2019.0,2019.0,March,0,107133.221477
1513,E11060,Employee_1060,50.0,Female,Finance,Finance Manager,181000.0,14.6,2024-06-10,Multan,...,No,115277.528090,65722.471910,57.012388,65722.471910,2024.0,2024.0,June,0,115277.528090


In [425]:
# “For each department, find the employee with the highest salary.”

In [426]:
rank=df.groupby("Department")["Salary"].rank(ascending=False)

In [427]:
df[rank ==1]

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Attrition,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month,Left,Dept_Avg_Salary
376,E10884,Employee_884,49.0,Male,HR,HR Manager,200700.0,18.1,2021-03-02,Multan,...,Yes,104615.200000,9.608480e+04,91.845927,9.608480e+04,2021.0,2021.0,March,1,104615.200000
490,E11086,Employee_1086,41.0,Male,Marketing,Marketing Manager,850000.0,9.0,2020-03-30,Islamabad,...,No,117392.857143,7.326071e+05,624.064497,7.326071e+05,2020.0,2020.0,March,0,117392.857143
1067,E10684,Employee_684,26.0,Female,Sales,Account Executive,1200000.0,1.8,2017-10-03,Karachi,...,No,112100.760456,1.087899e+06,970.465530,1.087899e+06,2017.0,2017.0,October,0,112100.760456
1078,E10179,Employee_179,50.0,Female,IT,Software Engineer,239400.0,20.4,2025-02-05,Multan,...,No,107133.221477,1.322668e+05,123.460096,1.322668e+05,2025.0,2025.0,February,0,107133.221477
1144,E11182,Employee_1182,49.0,Male,Operations,Coordinator,920000.0,16.8,2017-05-01,Lahore,...,Yes,113538.214286,8.064618e+05,710.299868,8.064618e+05,2017.0,2017.0,May,1,113538.214286
1284,E11349,Employee_1349,41.0,Female,Finance,Finance Manager,680000.0,11.8,2022-06-16,Karachi,...,No,115277.528090,5.647225e+05,489.880796,5.647225e+05,2022.0,2022.0,June,0,115277.528090
1401,E10476,Employee_476,48.0,Female,Customer Support,Support Specialist,236400.0,19.3,2020-03-15,Peshawar,...,No,103844.801980,1.325552e+05,127.647408,1.325552e+05,2020.0,2020.0,March,0,103844.801980


In [428]:
# “Which employees are earning more than the average salary of their job title?”

In [429]:
df["job_avg"]=df.groupby("Job_Title")["Salary"].transform("mean")

In [430]:
df[df["Salary"]>df["job_avg"]]

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Department_Avg_Salary,Salary_vs_Department_Avg,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month,Left,Dept_Avg_Salary,job_avg
0,E10674,Employee_674,43.0,Male,Operations,Coordinator,150100.0,13.9,2024-04-21,Peshawar,...,113538.214286,36561.785714,32.202185,36561.785714,2024.0,2024.0,April,0,113538.214286,124098.863636
1,E10204,Employee_204,34.0,Female,Finance,Finance Manager,142800.0,9.0,2021-07-31,Rawalpindi,...,115277.528090,27522.471910,23.874967,27522.471910,2021.0,2021.0,July,1,115277.528090,123723.913043
3,E11333,Employee_1333,35.0,Male,IT,IT Support Specialist,130000.0,8.9,2019-11-14,Islamabad,...,107133.221477,22866.778523,21.344246,22866.778523,2019.0,2019.0,November,0,107133.221477,105828.000000
6,E10723,Employee_723,36.0,Male,IT,Data Analyst,111100.0,6.5,2022-07-23,Lahore,...,107133.221477,3966.778523,3.702660,3966.778523,2022.0,2022.0,July,0,107133.221477,110902.747253
12,E10390,Employee_390,34.0,Female,Customer Support,Support Representative,146500.0,9.3,2020-02-16,Peshawar,...,103844.801980,42655.198020,41.075911,42655.198020,2020.0,2020.0,February,0,103844.801980,94255.072464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1509,E11222,Employee_1222,44.0,Male,Operations,Coordinator,193900.0,15.9,2020-07-14,Islamabad,...,113538.214286,80361.785714,70.779505,80361.785714,2020.0,2020.0,July,0,113538.214286,124098.863636
1510,E11093,Employee_1093,35.0,Female,Finance,Financial Analyst,122700.0,7.0,2021-11-26,Peshawar,...,115277.528090,7422.471910,6.438785,7422.471910,2021.0,2021.0,November,0,115277.528090,109112.121212
1511,E11193,Employee_1193,41.0,Male,IT,Software Engineer,166200.0,10.3,2019-03-29,Rawalpindi,...,107133.221477,59066.778523,55.133952,59066.778523,2019.0,2019.0,March,0,107133.221477,105522.535211
1513,E11060,Employee_1060,50.0,Female,Finance,Finance Manager,181000.0,14.6,2024-06-10,Multan,...,115277.528090,65722.471910,57.012388,65722.471910,2024.0,2024.0,June,0,115277.528090,123723.913043


In [431]:
# “For each department, calculate the percentage of employees who work overtime.”

In [432]:
overtime_avg=df[df["Overtime"]=="Yes"].groupby("Department").size()

In [433]:
total_ag=df.groupby("Department").size()

In [434]:
percenatge_overtime=(overtime_avg/total_ag)*100

In [435]:
percenatge_overtime

Department
Customer Support    28.712871
Finance             23.033708
HR                  29.600000
IT                  29.530201
Marketing           30.519481
Operations          28.928571
Sales               29.277567
dtype: float64

In [436]:
# “For each department, what percentage of employees were promoted?”

In [437]:
promoted_emp=df[df["Promotion"]=="Yes"].groupby("Department").size()

In [438]:
total_emp=df.groupby("Department").size()

In [439]:
percent_pro=(promoted_emp/total_emp)*100

In [440]:
percent_pro

Department
Customer Support    16.831683
Finance             17.977528
HR                  14.400000
IT                  19.798658
Marketing           20.129870
Operations          22.857143
Sales               17.870722
dtype: float64

# pd.cut()

In [441]:
df["Experience_Level"]=pd.cut(
     df["Years_Experience"],
    bins=[-1,2,5,10,float("inf")],
    labels=["Junior","Mid_Level","Senior","Expert"]
)

In [442]:
df["Experience_Level"]

0       Expert
1       Senior
2       Expert
3       Senior
4       Junior
         ...  
1510    Senior
1511    Expert
1512    Junior
1513    Expert
1514    Senior
Name: Experience_Level, Length: 1500, dtype: category
Categories (4, str): ['Junior' < 'Mid_Level' < 'Senior' < 'Expert']

# pd.qcut()

In [443]:
# Divide employees into 4 salary groups, from lowest-paid to highest-paid, with approximately 25% of employees in each group.

In [444]:
# “Divide employees into 4 equal-sized salary groups: Q1, Q2, Q3, Q4.”

In [445]:
df["Salary_Quartile"]=pd.qcut(
    df["Salary"],
    q=4,
    labels=["Q1","Q2","Q3","Q4"]
)

# pd.get_dummies() — categorical encoding.

In [446]:
# If you want to convert the Gender column into binary dummy variables, what Pandas code would you use?

In [447]:
pd.get_dummies(
    df["Gender"])


,Female,Male
0,False,True
1,True,False
2,False,True
3,False,True
4,True,False
...,...,...
1510,True,False
1511,False,True
1512,True,False
1513,True,False


# where() and mask()

In [448]:
# where(): Keep salaries above 80,000 unchanged, but replace salaries below 80,000 with NaN.
df["Salary"].where(df["Salary"]>80000)

0       150100.0
1       142800.0
2       101800.0
3       130000.0
4            NaN
          ...   
1510    122700.0
1511    166200.0
1512         NaN
1513    181000.0
1514    135800.0
Name: Salary, Length: 1500, dtype: float64

# combine_first()

In [449]:
# two columns containing alternative sources for the same information.

In [450]:
df13 = pd.DataFrame({
    "Phone_Primary": ["111", None, "333", None],
    "Phone_Secondary": [None, "222", "999", "444"]
})

In [451]:
(df13["Phone_Primary"]).combine_first(df13["Phone_Secondary"])

0    111
1    222
2    333
3    444
Name: Phone_Primary, dtype: str

# clip()

In [452]:
# clip() = keep values within a specified range; push values outside the range back to the boundary.

In [453]:
# df["Salary"].clip(lower=40000, upper=100000)

In [454]:
# Advanced date/time — extracting time periods

In [455]:
# "Was this order placed on a weekday or weekend?"
# df["Date"].dt.dayofweek

In [456]:
# "Was the order placed on a weekend?"
# df["Is_Weekend"] = df["Order_Date"].dt.dayofweek >= 5

# E-Commerce Sales & Customer Analytics : Project

In [457]:
df

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month,Left,Dept_Avg_Salary,job_avg,Experience_Level,Salary_Quartile
0,E10674,Employee_674,43.0,Male,Operations,Coordinator,150100.0,13.9,2024-04-21,Peshawar,...,32.202185,36561.785714,2024.0,2024.0,April,0,113538.214286,124098.863636,Expert,Q4
1,E10204,Employee_204,34.0,Female,Finance,Finance Manager,142800.0,9.0,2021-07-31,Rawalpindi,...,23.874967,27522.471910,2021.0,2021.0,July,1,115277.528090,123723.913043,Senior,Q4
2,E10308,Employee_308,41.0,Male,IT,Software Engineer,101800.0,10.1,2017-06-09,Peshawar,...,-4.978121,-5333.221477,2017.0,2017.0,June,0,107133.221477,105522.535211,Expert,Q2
3,E11333,Employee_1333,35.0,Male,IT,IT Support Specialist,130000.0,8.9,2019-11-14,Islamabad,...,21.344246,22866.778523,2019.0,2019.0,November,0,107133.221477,105828.000000,Senior,Q3
4,E10492,Employee_492,22.0,Female,IT,Data Analyst,35000.0,0.0,2023-01-16,Islamabad,...,-67.330395,-72133.221477,2023.0,2023.0,January,0,107133.221477,110902.747253,Junior,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1510,E11093,Employee_1093,35.0,Female,Finance,Financial Analyst,122700.0,7.0,2021-11-26,Peshawar,...,6.438785,7422.471910,2021.0,2021.0,November,0,115277.528090,109112.121212,Senior,Q3
1511,E11193,Employee_1193,41.0,Male,IT,Software Engineer,166200.0,10.3,2019-03-29,Rawalpindi,...,55.133952,59066.778523,2019.0,2019.0,March,0,107133.221477,105522.535211,Expert,Q4
1512,E11210,Employee_1210,24.0,Female,Sales,Account Executive,51600.0,1.4,2018-09-06,Lahore,...,-53.969982,-60500.760456,2018.0,2018.0,September,0,112100.760456,120481.578947,Junior,Q1
1513,E11060,Employee_1060,50.0,Female,Finance,Finance Manager,181000.0,14.6,2024-06-10,Multan,...,57.012388,65722.471910,2024.0,2024.0,June,0,115277.528090,123723.913043,Expert,Q4


In [458]:
df.shape

(1500, 31)

In [459]:
df.head(3)

,Employee_ID,Name,Age,Gender,Department,Job_Title,Salary,Years_Experience,Join_Date,City,...,Salary_vs_Department_Avg_Pct,difference_salary,year_specific,join_Year,Join_Month,Left,Dept_Avg_Salary,job_avg,Experience_Level,Salary_Quartile
0,E10674,Employee_674,43.0,Male,Operations,Coordinator,150100.0,13.9,2024-04-21,Peshawar,...,32.202185,36561.785714,2024.0,2024.0,April,0,113538.214286,124098.863636,Expert,Q4
1,E10204,Employee_204,34.0,Female,Finance,Finance Manager,142800.0,9.0,2021-07-31,Rawalpindi,...,23.874967,27522.471910,2021.0,2021.0,July,1,115277.528090,123723.913043,Senior,Q4
2,E10308,Employee_308,41.0,Male,IT,Software Engineer,101800.0,10.1,2017-06-09,Peshawar,...,-4.978121,-5333.221477,2017.0,2017.0,June,0,107133.221477,105522.535211,Expert,Q2


In [460]:
audit = {
    "Rows":df.shape[0],
    "Columns":df.shape[1],
    "Missing":df.isna().sum().sum(),
    "Duplicates":df.duplicated().sum()
}

In [461]:
audit_df = pd.DataFrame([audit])

In [462]:
audit_df

,Rows,Columns,Missing,Duplicates
0,1500,31,59,0


In [463]:
# Which department has the highest employee attrition rate?"

In [464]:
A=df[df["Attrition"]=="Yes"].groupby("Department").size()

In [465]:
T=df.groupby("Department").size()

In [466]:
A_rate=(A/T)*100

In [467]:
A_rate.nlargest(1)

Department
Sales    12.1673
dtype: float64

In [468]:
# "Is employee attrition higher among males or females?"

In [469]:
A_Gender=df[df["Attrition"]=="Yes"].groupby("Gender").size()

In [470]:
T=df.groupby("Gender").size()

In [471]:
A_rate_Gender=(A_Gender/T)*100

In [472]:
A_rate_Gender.nlargest(1)

Gender
Female    10.80292
dtype: float64

In [473]:
# "Among employees who work overtime and those who don't, which group has the higher attrition rate?"

In [474]:
A_overtime=df[df["Attrition"]=="Yes"].groupby("Overtime").size()

In [475]:
overtime_T=df.groupby("Overtime").size()

In [476]:
Attri_overtime=(A_overtime/overtime_T)*100

In [477]:
Attri_overtime.nlargest(1)

Overtime
Yes    13.519814
dtype: float64

In [478]:
# Attrition by Department + Overtime
# "Within each department, does overtime status affect attrition?"

In [479]:
AD_overtime=df[df["Attrition"]=="Yes"].groupby(["Department","Overtime"]).size()

In [480]:
overtime_D=df.groupby(["Department","Overtime"]).size()

In [481]:
ADO_rate=(AD_overtime/overtime_D)*100

In [482]:
ADO_rate.nlargest(1)

Department  Overtime
IT          Yes         18.181818
dtype: float64

In [483]:
# “Does remote work status affect employee attrition?”

In [484]:
s=df[df["Attrition"]=="Yes"].groupby("Remote_Work").size()

In [485]:
t=df.groupby("Remote_Work").size()

In [486]:
remote_attri=(s/t)*100

In [487]:
remote_attri.nlargest(1)

Remote_Work
Yes    10.921502
dtype: float64

In [488]:
# “Which experience group has the highest attrition rate?”

In [489]:
E_L=df[df["Attrition"]=="Yes"].groupby("Experience_Level").size()

In [490]:
E_T=df.groupby("Experience_Level").size()

In [491]:
A_E=(E_L/E_T)*100

In [492]:
A_E.nlargest(1)

Experience_Level
Mid_Level    13.355049
dtype: float64

In [493]:
#"Do employees with higher salaries have lower attrition?"

In [494]:
df["Salary"].max()

np.float64(1200000.0)

In [495]:
df["Salary"].min()

np.float64(35000.0)

In [496]:
df["Salary"].describe()

count    1.500000e+03
mean     1.105669e+05
std      6.322907e+04
min      3.500000e+04
25%      7.410000e+04
50%      1.064000e+05
75%      1.357000e+05
max      1.200000e+06
Name: Salary, dtype: float64

In [497]:
df["Salary_Group"]=pd.qcut(
    df["Salary"],
    q=4,
    labels=["Q1","Q2","Q3","Q4"]
)

In [498]:
df["Salary_Group"]

0       Q4
1       Q4
2       Q2
3       Q3
4       Q1
        ..
1510    Q3
1511    Q4
1512    Q1
1513    Q4
1514    Q4
Name: Salary_Group, Length: 1500, dtype: category
Categories (4, str): ['Q1' < 'Q2' < 'Q3' < 'Q4']

In [499]:
A_S=df[df["Attrition"]=="Yes"].groupby("Salary_Group").size()

In [500]:
T_S=df.groupby("Salary_Group").size()

In [501]:
AS_Rate=(A_S/T_S)*100

In [502]:
AS_Rate.nlargest(1)

Salary_Group
Q1    11.968085
dtype: float64

In [503]:
# "Do employees who received a promotion have a different attrition rate from employees who did not?"

In [504]:
A_P=df[df["Attrition"]=="Yes"].groupby("Promotion").size()

In [505]:
T_P=df.groupby("Promotion").size()

In [506]:
AP_Rate=(A_P/T_P)*100

In [507]:
AP_Rate.nlargest(1)

Promotion
No    10.617284
dtype: float64

In [508]:
# Do employees with lower performance scores have higher attrition?

In [509]:
df["Performance_Score"].value_counts().sort_index()

Performance_Score
1.0     74
2.0    211
3.0    531
4.0    500
5.0    184
Name: count, dtype: int64

In [510]:
AT_P=df[df["Attrition"]=="Yes"].groupby("Performance_Score").size()

In [511]:
TP=df.groupby("Performance_Score").size()

In [512]:
AT_rate=(AT_P/TP)*100

In [513]:
AT_rate.nlargest(1)

Performance_Score
1.0    14.864865
dtype: float64

In [514]:
# Do employees with lower satisfaction scores have higher attrition?

In [515]:
A_S=df[df["Attrition"]=="Yes"].groupby("Satisfaction_Score").size()

In [516]:
T_S=df.groupby("Satisfaction_Score").size()

In [517]:
Attrition_sat=(A_S/T_S)*100

In [518]:
Attrition_sat.nlargest(1)

Satisfaction_Score
1.0    19.047619
dtype: float64

In [519]:
# Does average salary differ across experience levels?

In [520]:
df.groupby("Experience_Level")["Salary"].mean()

Experience_Level
Junior        57397.477064
Mid_Level     79309.771987
Senior       111473.916811
Expert       162485.301508
Name: Salary, dtype: float64

In [521]:
# Which department has the highest average salary, and which has the lowest average salary?

In [522]:
dep_salary=df.groupby("Department")["Salary"].mean()

In [523]:
dep_salary.nlargest(1)

Department
Marketing    117392.857143
Name: Salary, dtype: float64

In [524]:
dep_salary.nsmallest(1)

Department
Customer Support    103844.80198
Name: Salary, dtype: float64

In [525]:
# Which department has the highest promotion rate?

In [526]:
D_P=df[df["Promotion"]=="Yes"].groupby("Department").size()

In [527]:
DEP_TOTAL=df.groupby("Department").size()

In [528]:
PRO_Rate=(D_P/DEP_TOTAL)*100

In [529]:
PRO_Rate

Department
Customer Support    16.831683
Finance             17.977528
HR                  14.400000
IT                  19.798658
Marketing           20.129870
Operations          22.857143
Sales               17.870722
dtype: float64

In [530]:
PRO_Rate.nlargest(1)

Department
Operations    22.857143
dtype: float64

In [531]:
# Which department has the highest average training hours?

In [532]:
T_D=df.groupby("Department")["Training_Hours"].mean()

In [533]:
T_D.nlargest(1)

Department
HR    24.576
Name: Training_Hours, dtype: float64

In [534]:
# Do employees who receive more training have lower attrition?

In [535]:
df["Training_Group"]=pd.qcut(
    df["Training_Hours"],
    q=3,
    labels=["Low_training","Medium_training","High_training"]
)

In [536]:
ATT_T=df[df["Attrition"]=="Yes"].groupby("Training_Group").size()

In [537]:
Total_TG=df.groupby("Training_Group").size()

In [538]:
AT_Rate=(ATT_T/Total_TG)*100

In [539]:
AT_Rate.nlargest(1)

Training_Group
High_training    10.920771
dtype: float64

In [540]:
# Do employees who complete fewer projects have higher attrition?

In [541]:
df["Projects_Completed"].max()

np.int64(13)

In [542]:
df["Projects_Completed"].min()

np.int64(0)

In [543]:
df["Projects_Group"]=pd.qcut(
    df["Projects_Completed"],
    q=3,
    labels=["Few_pro","Med_pro","more_pro"]
)
    

In [544]:
AA=df[df["Attrition"]=="Yes"].groupby("Projects_Group").size()

In [545]:
PP=df.groupby("Projects_Group").size()

In [546]:
Pro_Att=(AA/PP)*100

In [547]:
Pro_Att.nlargest(1)

Projects_Group
Med_pro    10.752688
dtype: float64

In [548]:
# Do employees with more leave days have higher attrition?

In [549]:
df["Leave_Days"].max()

np.int64(29)

In [550]:
df["Leave_Days"].min()

np.int64(0)

In [551]:
df["Leave_Days"].describe()

count    1500.000000
mean       11.917333
std         4.975574
min         0.000000
25%         9.000000
50%        12.000000
75%        15.000000
max        29.000000
Name: Leave_Days, dtype: float64

In [552]:
df["Leave_Group"]=pd.qcut(
    df["Leave_Days"],
    q=3,
    labels=["Low_leave","Med_leave","High_leave"]
)

In [553]:
L_G=df[df["Attrition"]=="Yes"].groupby("Leave_Group").size()

In [554]:
L_T=df.groupby("Leave_Group").size()

In [555]:
Leave_ATT=(L_G/L_T)*100

In [556]:
Leave_ATT.nlargest(1)

Leave_Group
Med_leave    10.344828
dtype: float64

In [557]:
# Within each Overtime group, what is the average salary?

In [558]:
df.groupby("Overtime")["Salary"].mean()

Overtime
No     108642.717087
Yes    115370.629371
Name: Salary, dtype: float64

In [559]:
# What is the average salary for each Gender within each Department?

In [560]:
df.groupby(["Department","Gender"])["Salary"].mean()

Department        Gender
Customer Support  Female    102091.037736
                  Male      105781.250000
Finance           Female    115600.000000
                  Male      114962.222222
HR                Female     93891.379310
                  Male      113898.507463
IT                Female    108878.623188
                  Male      105627.812500
Marketing         Female    120258.653846
                  Male      115931.862745
Operations        Female    107674.793388
                  Male      118000.314465
Sales             Female    114612.295082
                  Male      109927.659574
Name: Salary, dtype: float64

In [561]:
# Within each department and gender, what is the attrition rate?

In [562]:
ATT_G=df[df["Attrition"]=="Yes"].groupby(["Department","Gender"]).size()

In [563]:
Total_G=df.groupby(["Department","Gender"]).size()

In [564]:
ATT_DG=(ATT_G/Total_G)*100

In [565]:
ATT_DG.nlargest(1)

Department  Gender
IT          Male      13.75
dtype: float64

In [566]:
# What is the average salary for each Experience Level within each Department?

In [567]:
df.groupby(["Department","Experience_Level"])["Salary"].mean()

Department        Experience_Level
Customer Support  Junior               51937.837838
                  Mid_Level            75213.636364
                  Senior              111223.529412
                  Expert              154383.962264
Finance           Junior               52076.923077
                  Mid_Level            71503.030303
                  Senior              112377.777778
                  Expert              173678.571429
HR                Junior               47895.238095
                  Mid_Level            82986.363636
                  Senior              105822.448980
                  Expert              153336.363636
IT                Junior               53627.777778
                  Mid_Level            75757.575758
                  Senior              106933.043478
                  Expert              156762.962963
Marketing         Junior               58170.454545
                  Mid_Level            79979.166667
                  Senior     

In [568]:
# Within each Department and Experience Level, what is the attrition rate?

In [569]:
ATT_EX=df[df["Attrition"]=="Yes"].groupby(["Department","Experience_Level"]).size()

In [570]:
T_EX=df.groupby(["Department","Experience_Level"]).size()

In [571]:
Att_Rate=(ATT_EX/T_EX)*100

In [572]:
Att_Rate.nlargest(1)

Department  Experience_Level
IT          Expert              19.753086
dtype: float64

In [573]:
# Create a table showing the average salary for each Department × Gender combination.

In [574]:
avg_salary=df.pivot_table(
    index="Department",
    columns="Gender",
    values="Salary",
    aggfunc="mean"
)

In [575]:
avg_salary

Gender,Female,Male
Department,,
Customer Support,102091.037736,105781.250000
Finance,115600.000000,114962.222222
HR,93891.379310,113898.507463
IT,108878.623188,105627.812500
Marketing,120258.653846,115931.862745
Operations,107674.793388,118000.314465
Sales,114612.295082,109927.659574


In [576]:
# Create a table showing the attrition rate (%) for each Department × Gender combination.

In [577]:
df["Attrion_rate"]=np.where(df["Attrition"]=="Yes",1,0)

In [578]:
AT_GEN=df.pivot_table(
    index="Department",
    columns="Gender",
    values="Attrion_rate",
    aggfunc="mean"
)

In [579]:
AT_DEN=AT_GEN*100

In [580]:
AT_GEN

Gender,Female,Male
Department,,
Customer Support,0.132075,0.072917
Finance,0.102273,0.133333
HR,0.034483,0.044776
IT,0.094203,0.137500
Marketing,0.115385,0.029412
Operations,0.123967,0.094340
Sales,0.122951,0.120567


In [581]:
# Which Department + Gender combination has the highest attrition rate?

In [582]:
AT_GEN.stack().nlargest(1)

Department  Gender
IT          Male      0.1375
dtype: float64

In [583]:
# Which department has the highest average Performance Score, and what is its average score?

In [584]:
Dep_Score=df.groupby("Department")["Performance_Score"].mean()

In [585]:
Dep_Score.nlargest(1)

Department
Operations    3.453571
Name: Performance_Score, dtype: float64

In [586]:
# Which department has the highest average number of Projects Completed per employee?

In [587]:
Dep_Project=df.groupby("Department")["Projects_Completed"].mean()

In [588]:
Dep_Project.nlargest(1)

Department
Finance    5.269663
Name: Projects_Completed, dtype: float64

In [589]:
# Which department has the highest average Projects Completed among employees who left the company?

In [590]:
Attri_Pro=df[df["Attrition"]=="Yes"].groupby("Department")["Projects_Completed"].mean()

In [591]:
Attri_Pro.nlargest(1)

Department
Customer Support    5.619048
Name: Projects_Completed, dtype: float64

In [592]:
# Among employees who left the company, which department has the lowest average Performance Score?

In [593]:
Attri_Score=df[df["Attrition"]=="Yes"].groupby("Department")["Performance_Score"].mean()

In [594]:
Attri_Score.nsmallest(1)

Department
HR    2.6
Name: Performance_Score, dtype: float64

In [595]:
# For each department, calculate both the average Salary and average Performance Score.

In [596]:
dep_salary_pro=df.groupby("Department").agg(
    avg_Salary=("Salary","mean"),
    avg_Performance_Score=("Performance_Score","mean")
)

In [597]:
dep_sal=df.groupby("Department")["Salary"].mean()

In [598]:
dep_sal.nlargest(1)

Department
Marketing    117392.857143
Name: Salary, dtype: float64

In [599]:
dep_sal.nsmallest(1)

Department
Customer Support    103844.80198
Name: Salary, dtype: float64

In [600]:
dep_sal_emp=df.groupby("Department").agg(
    avg_Salary=("Salary","mean"),
    avg_Performance_Score=("Performance_Score","mean"),
    no_employee=("Name","size")
)

In [601]:
dep_sal_emp

,avg_Salary,avg_Performance_Score,no_employee
Department,,,
Customer Support,103844.801980,3.420792,202
Finance,115277.528090,3.264045,178
HR,104615.200000,3.144000,125
IT,107133.221477,3.385906,298
Marketing,117392.857143,3.227273,154
Operations,113538.214286,3.453571,280
Sales,112100.760456,3.311787,263


In [602]:
# Which department has the highest average Performance Score?

In [603]:
dep_sal_emp["avg_Performance_Score"].nlargest(1)

Department
Operations    3.453571
Name: avg_Performance_Score, dtype: float64

In [604]:
# Show all departments ranked from highest to lowest average salary.

In [605]:
dep_sal_emp.sort_values(by="avg_Salary",ascending=False)

,avg_Salary,avg_Performance_Score,no_employee
Department,,,
Marketing,117392.857143,3.227273,154
Finance,115277.528090,3.264045,178
Operations,113538.214286,3.453571,280
Sales,112100.760456,3.311787,263
IT,107133.221477,3.385906,298
HR,104615.200000,3.144000,125
Customer Support,103844.801980,3.420792,202


In [606]:
dep_sal_emp.sort_values(by="avg_Performance_Score",ascending=False)

,avg_Salary,avg_Performance_Score,no_employee
Department,,,
Operations,113538.214286,3.453571,280
Customer Support,103844.801980,3.420792,202
IT,107133.221477,3.385906,298
Sales,112100.760456,3.311787,263
Finance,115277.528090,3.264045,178
Marketing,117392.857143,3.227273,154
HR,104615.200000,3.144000,125


In [607]:
dep_sal_emp.sort_values(by=["avg_Performance_Score","avg_Salary"],ascending=[False,False])

,avg_Salary,avg_Performance_Score,no_employee
Department,,,
Operations,113538.214286,3.453571,280
Customer Support,103844.801980,3.420792,202
IT,107133.221477,3.385906,298
Sales,112100.760456,3.311787,263
Finance,115277.528090,3.264045,178
Marketing,117392.857143,3.227273,154
HR,104615.200000,3.144000,125


In [608]:
# Using dep_sal_emp:

# Add a new column called Salary_Rank that ranks departments according to their average salary, where 1 = highest average salary.

In [610]:
dep_sal_emp["Salary_Rank"]=dep_sal_emp["avg_Salary"].rank(ascending=False,method="min")

In [611]:
dep_sal_emp["Salary_Rank"]

Department
Customer Support    7.0
Finance             2.0
HR                  6.0
IT                  5.0
Marketing           1.0
Operations          3.0
Sales               4.0
Name: Salary_Rank, dtype: float64

In [614]:
# Add a column called Performance_Rank where the department with the highest average Performance Score gets rank 1.

In [615]:
dep_sal_emp["Performance_Rank"]=dep_sal_emp["avg_Performance_Score"].rank(ascending=False,method="min")

In [616]:
# Find departments that have average salary greater than 100,000 AND average performance score greater than 3.5.

In [620]:
dep_sal_emp[(dep_sal_emp["avg_Salary"]>100000)&(dep_sal_emp["avg_Performance_Score"]>3.5)]

,avg_Salary,avg_Performance_Score,no_employee,Salary_Rank,Performance_Rank
Department,,,,,


In [628]:
dep_sal_emp=df.groupby("Department").agg(
    avg_Salary=("Salary","mean"),
    avg_Performance_Score=("Performance_Score","mean"),
    no_employee=("Name","size"),
    avg_pro_completed=("Projects_Completed","mean")
)

In [631]:
dep_sal_emp["Salary per Project"]=dep_sal_emp["avg_Salary"]/dep_sal_emp["avg_pro_completed"]

In [632]:
# Which department has the lowest Salary per Project?

In [634]:
dep_sal_emp["Salary per Project"].nsmallest(1)

Department
HR    20056.595092
Name: Salary per Project, dtype: float64

In [639]:
dep_sal_emp[(dep_sal_emp["avg_Salary"]>100000)&(dep_sal_emp["avg_Performance_Score"]>3.5)&(dep_sal_emp["avg_pro_completed"]>6)].sort_values(by="avg_Salary",ascending=False)

,avg_Salary,avg_Performance_Score,no_employee,avg_pro_completed,Salary per Project
Department,,,,,


In [645]:
df[df["Attrition"]=="Yes"].groupby("Department").agg(
    avg_salary=("Salary","mean"),
    avg_perform_score=("Performance_Score","mean"),
    avg_pro_completed=("Projects_Completed","mean")
)                      

,avg_salary,avg_perform_score,avg_pro_completed
Department,,,
Customer Support,85285.714286,3.238095,5.619048
Finance,99385.714286,3.380952,5.095238
HR,130240.000000,2.600000,3.200000
IT,116125.714286,3.228571,4.028571
Marketing,135044.444444,2.888889,5.222222
Operations,137086.666667,3.400000,4.900000
Sales,111606.250000,3.250000,4.812500
